# 02 Predictions And External Evaluation

Hand-maintained revision runbook. Do not regenerate from `scripts/revision/create_revision_notebooks.py` unless that generator is first updated to the fixed-epoch final_ema/model_runs protocol.

**Full reviewer-evidence profile:** use an **A100 High-RAM** runtime for OOF/external inference, PTB-XL fold 9, learned external comparators, and representation extraction. CPU-only gate/bootstrap/head-fitting stages can run later in a fresh CPU runtime. A fresh runtime may intentionally restart after dependency repair; after reconnecting, rerun from Setup. Every expensive stage uses provenance-checked artifacts or fold caches under `/content/drive/MyDrive/ECG-Ramba/revision_artifacts/reports/revision`, and command logs are streamed both locally and to that canonical Drive root. The Drive checkout at `ECG-Ramba/ECG-RAMBA/reports/revision` is legacy/audit-only and is never a reuse source.

This notebook completes the external/zero-target-label/few-shot portion of the reviewer package. Learned ResNet/Raw-Mamba/Transformer OOF summaries and checkpoints must first be completed by Notebook 04; calibration/reliability, robustness, pooling, and representation audits remain owned by Notebooks 03, 05, and 06 respectively.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import shutil
import zipfile

DRIVE_MOUNT = Path('/content/drive')
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'ECG-Ramba'
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'

def _drive_root_ready(root):
    try:
        return root.is_dir() and any(root.iterdir())
    except Exception:
        return False

try:
    from google.colab import drive
    if not _drive_root_ready(DRIVE_ROOT):
        try:
            drive.mount(str(DRIVE_MOUNT))
        except Exception as exc:
            print(f'Drive mount initial attempt failed or mountpoint was stale: {exc}')
            print('Retrying Drive mount with force_remount=True ...')
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
    else:
        print('Drive root already visible:', DRIVE_ROOT)
except Exception as exc:
    print(f'Drive mount skipped or unavailable: {exc}')

if not _drive_root_ready(DRIVE_ROOT):
    visible_mount = sorted(path.name for path in DRIVE_MOUNT.iterdir()) if DRIVE_MOUNT.exists() else []
    visible_my_drive = sorted(path.name for path in (DRIVE_MOUNT / 'MyDrive').iterdir()) if (DRIVE_MOUNT / 'MyDrive').exists() else []
    raise RuntimeError(
        f'Google Drive root is not visible at {DRIVE_ROOT}. '
        f'Visible under {DRIVE_MOUNT}: {visible_mount}; visible under MyDrive: {visible_my_drive}. '
        'In Colab, run Runtime > Disconnect and delete runtime, then rerun this setup cell; '
        'or manually run drive.mount("/content/drive", force_remount=True). '
        'Do not continue while the ECG-Ramba Drive folder lists as empty.'
    )

REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path(os.environ.get('ECG_RAMBA_REPO_DIR', '/content/ECG-RAMBA'))
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
chapman_candidates = [
    DRIVE_ROOT / 'WFDB-ChapmanShaoxing.zip',
    DRIVE_ROOT / 'WFDB_ChapmanShaoxing.zip',
    DRIVE_ROOT / 'chapman.zip',
    DRIVE_ROOT / 'archive.zip',
]
chapman_zip = next((path for path in chapman_candidates if path.is_file()), None)
if chapman_zip is None:
    visible_zips = sorted(path.name for path in DRIVE_ROOT.glob('*.zip')) if DRIVE_ROOT.exists() else []
    raise FileNotFoundError(
        'Chapman dataset ZIP was not found. Checked: '
        + ', '.join(str(path) for path in chapman_candidates)
        + f'. ZIP files currently visible under {DRIVE_ROOT}: {visible_zips}'
    )
if chapman_zip.stat().st_size == 0 or not zipfile.is_zipfile(chapman_zip):
    raise ValueError(f'Chapman dataset is not a valid non-empty ZIP: {chapman_zip}')
model_pointer = DRIVE_ROOT / 'model_runs' / 'current_final_ema_model_dir.txt'
model_dir_env = os.environ.get('ECG_RAMBA_MODEL_DIR')
if model_dir_env:
    model_dir = Path(model_dir_env).expanduser()
elif model_pointer.is_file():
    model_dir = Path(model_pointer.read_text(encoding='utf-8').strip()).expanduser()
else:
    model_dir = DRIVE_ROOT / 'model'
if not model_dir.is_dir():
    raise FileNotFoundError(f'Model directory not found: {model_dir}. Run notebook 02a first or set ECG_RAMBA_MODEL_DIR.')
os.environ['ECG_RAMBA_CHAPMAN_ZIP'] = str(chapman_zip)
os.environ['ECG_RAMBA_MODEL_DIR'] = str(model_dir)
print('Model dir:', model_dir)
print('Model pointer:', model_pointer, 'exists=', model_pointer.is_file())
os.environ['ECG_RAMBA_LOCAL_ROOT'] = str(LOCAL_RUNTIME_ROOT)
os.environ['ECG_RAMBA_EXTRACT_DIR'] = str(LOCAL_RUNTIME_ROOT / 'chapman')
os.environ['ECG_RAMBA_USE_CLEAN_CACHE'] = '1'
os.environ['ECG_RAMBA_SAVE_CLEAN_CACHE'] = '0'

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids touching Drive-backed working trees while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, remove /content/ECG-RAMBA or set ECG_RAMBA_REPO_DIR.')
        print('=' * 80)
        if os.environ.get('ECG_RAMBA_ALLOW_STALE_REPO', '0') != '1':
            raise RuntimeError('git pull failed; refusing to continue with stale code. Set ECG_RAMBA_ALLOW_STALE_REPO=1 only for debugging.')
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)

# BEGIN FORENSIC CODE AUTHORITY PIN
FORENSIC_CODE_AUTHORITY_CAPABILITY = 'canonical_git_commit_pin_v1'
FORENSIC_CODE_AUTHORITY_SCHEMA_VERSION = 1
_AUTHORITY_BOOTSTRAP_ALLOWED = False

def _pin_forensic_code_authority():
    import json as _authority_json
    import os as _authority_os
    import re as _authority_re
    import subprocess as _authority_subprocess
    import uuid as _authority_uuid
    from datetime import datetime as _authority_datetime, timezone as _authority_timezone
    from pathlib import Path as _AuthorityPath

    manifest_path = _AuthorityPath(MIRROR_REVISION_ROOT) / 'manifests' / 'notebook_code_authority.json'
    requested_commit = _authority_os.environ.get('ECG_RAMBA_AUTHORITY_COMMIT', '').strip().lower()
    reset_requested = _authority_os.environ.get('ECG_RAMBA_RESET_CODE_AUTHORITY', '0') == '1'
    rotate_to_branch_head = (
        _AUTHORITY_BOOTSTRAP_ALLOWED
        and _authority_os.environ.get('ECG_RAMBA_ROTATE_CODE_AUTHORITY_TO_BRANCH_HEAD', '1') == '1'
        and not requested_commit
        and not reset_requested
    )
    commit_pattern = _authority_re.compile(r'[0-9a-f]{40}')

    def git(*args, check=True):
        result = _authority_subprocess.run(
            ['git', *args],
            cwd=str(REPO_DIR),
            check=False,
            text=True,
            stdout=_authority_subprocess.PIPE,
            stderr=_authority_subprocess.STDOUT,
        )
        if check and result.returncode:
            raise RuntimeError(
                'Code-authority git command failed: git '
                + ' '.join(args)
                + '\n'
                + (result.stdout or '')[-4000:]
            )
        return result

    if rotate_to_branch_head:
        requested_commit = git('rev-parse', 'HEAD').stdout.strip().lower()
        reset_requested = True
        print('Notebook 00 authority rotation target:', requested_commit)

    if reset_requested and not _AUTHORITY_BOOTSTRAP_ALLOWED:
        raise RuntimeError(
            'Only Notebook 00 may rotate the canonical code authority. '
            'Run Notebook 00 with ECG_RAMBA_RESET_CODE_AUTHORITY=1 and an explicit '
            'ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    if reset_requested and not commit_pattern.fullmatch(requested_commit):
        raise RuntimeError(
            'Authority reset requires ECG_RAMBA_AUTHORITY_COMMIT as a full 40-character git SHA.'
        )

    manifest = None
    if manifest_path.is_file() and not reset_requested:
        manifest = _authority_json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest.get('capability') != 'canonical_git_commit_pin_v1':
            raise RuntimeError('Canonical code-authority manifest capability is invalid.')
        if int(manifest.get('schema_version', 0)) != 1:
            raise RuntimeError('Canonical code-authority manifest schema is invalid.')
        expected_commit = str(manifest.get('git_commit', '')).strip().lower()
        if not commit_pattern.fullmatch(expected_commit):
            raise RuntimeError('Canonical code-authority manifest lacks a full git SHA.')
        if str(manifest.get('repository_url', '')).rstrip('/') != str(REPO_URL).rstrip('/'):
            raise RuntimeError('Canonical code-authority repository URL differs from this notebook.')
        if str(manifest.get('branch', '')) != str(BRANCH):
            raise RuntimeError('Canonical code-authority branch differs from this notebook runtime.')
        if requested_commit and requested_commit != expected_commit:
            raise RuntimeError(
                'ECG_RAMBA_AUTHORITY_COMMIT differs from the canonical authority manifest. '
                'Rotate authority explicitly in Notebook 00; do not override it in a downstream notebook.'
            )
    else:
        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            raise FileNotFoundError(
                'Canonical code-authority manifest is missing. Run Notebook 00 first in a fresh runtime; '
                'downstream notebooks fail closed instead of following a moving branch.'
            )
        expected_commit = requested_commit or git('rev-parse', 'HEAD').stdout.strip().lower()
        if not commit_pattern.fullmatch(expected_commit):
            raise RuntimeError('Notebook 00 could not resolve a full code-authority git SHA.')

    tracked_status = git('status', '--porcelain', '--untracked-files=no').stdout.strip()
    if tracked_status:
        raise RuntimeError(
            'Tracked files differ from git before authority checkout. Use a fresh Colab clone; '
            'authority pinning will not stash or overwrite local edits.\n' + tracked_status[:4000]
        )

    fetch = git('fetch', 'origin', '--prune', check=False)
    if fetch.returncode:
        print('WARNING: git fetch failed; accepting only an already-present pinned commit.')
        print((fetch.stdout or '')[-2000:])
    git('cat-file', '-e', expected_commit + '^{commit}')
    git('checkout', '--detach', expected_commit)
    observed_commit = git('rev-parse', 'HEAD').stdout.strip().lower()
    if observed_commit != expected_commit:
        raise RuntimeError(
            f'Code-authority checkout mismatch: expected={expected_commit} observed={observed_commit}'
        )

    if manifest is None or reset_requested:
        manifest = {
            'capability': 'canonical_git_commit_pin_v1',
            'schema_version': 1,
            'git_commit': expected_commit,
            'repository_url': str(REPO_URL),
            'branch': str(BRANCH),
            'established_utc': _authority_datetime.now(_authority_timezone.utc).isoformat(),
            'established_by': '00_colab_bootstrap.ipynb',
            'selection': (
                'fetched_branch_head_at_notebook00_rotation'
                if rotate_to_branch_head
                else ('explicit_environment_sha' if requested_commit else 'fetched_branch_head_at_initial_bootstrap')
            ),
        }
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = manifest_path.with_name(
            manifest_path.name + '.partial.' + _authority_uuid.uuid4().hex
        )
        with temporary.open('w', encoding='utf-8') as handle:
            handle.write(_authority_json.dumps(manifest, indent=2, sort_keys=True) + '\n')
            handle.flush()
            _authority_os.fsync(handle.fileno())
        _authority_os.replace(temporary, manifest_path)
        print('Established canonical code authority:', manifest_path)

    _authority_os.environ['ECG_RAMBA_AUTHORITY_COMMIT'] = expected_commit
    _authority_os.environ.pop('ECG_RAMBA_RESET_CODE_AUTHORITY', None)
    globals()['CODE_AUTHORITY_MANIFEST_PATH'] = manifest_path
    globals()['CODE_AUTHORITY'] = manifest
    print('Pinned code authority:', expected_commit)
    print('Authority manifest   :', manifest_path)
    return manifest

CODE_AUTHORITY = _pin_forensic_code_authority()
# END FORENSIC CODE AUTHORITY PIN

_run_setup('git status --short --branch', check=False)

# BEGIN FORENSIC NOTEBOOK 02 CAPABILITY PREFLIGHT
# The immutable authority checkout is the source of code. Validate reviewed,
# machine-readable capabilities without downloading mutable branch files into
# the detached checkout.
import ast as _compat_ast

REVISION_CAPABILITY_REQUIREMENTS = {
    'scripts/revision/03_generate_external_predictions.py': {
        'NOTEBOOK_02_EXTERNAL_EXPORT_CAPABILITY': 'external_export_full10s_grouped_v1',
        'NOTEBOOK_02_EXTERNAL_EXPORT_SCHEMA_VERSION': 1,
    },
    'scripts/revision/18_external_protocol_gate.py': {
        'NOTEBOOK_02_EXTERNAL_GATE_CAPABILITY': 'external_gate_full10s_grouped_v1',
        'NOTEBOOK_02_EXTERNAL_GATE_SCHEMA_VERSION': 1,
    },
    'scripts/revision/external_reuse_contract.py': {
        'EXTERNAL_REUSE_CAPABILITY': 'source_bound_external_reuse_v1',
        'EXTERNAL_REUSE_SCHEMA_VERSION': 1,
    },
    'scripts/revision/49_build_oof_group_sidecar.py': {
        'GROUP_SIDECAR_CAPABILITY': 'chapman_oof_group_sidecar_v1',
        'GROUP_SIDECAR_SCHEMA_VERSION': 1,
    },
}
REVISION_TOKEN_REQUIREMENTS = {
    'scripts/revision/common.py': [
        'one-label mapped task retains positive-label',
        'average="binary"',
    ],
    'scripts/revision/artifact_mirror.py': [
        '--verify-existing',
        '--include-prefix',
        'discovered_unmanifested_count',
        'recoverable_publish_transaction_v1',
        'PUBLISH_TRANSACTION_NAME',
    ],
    'scripts/revision/01_generate_predictions.py': [
        '--fold-cache-dir',
        'OOF_CACHE_PROVENANCE_SCHEMA_VERSION',
        'cache_contract_sha256',
    ],
    'scripts/revision/03_generate_external_predictions.py': [
        '--georgia-mapping-review',
        '--georgia-code-inventory-out',
        '--cpsc-annotation-audit-out',
        'validate_checkpoint_files_against_oof_run_manifest',
    ],
    'scripts/revision/06_freeze_oof.py': [
        '--check-existing-freeze',
        'Validated without rewrite',
        '--metadata-refresh-from-existing-oof',
        'verified_metadata_only_refresh',
        '--manuscript-ready-strict',
        'source_archive_sha256',
    ],
    'scripts/revision/18_external_protocol_gate.py': [
        'georgia_mapping_inventory',
        'cpsc_annotation_audit',
        'metric_implementation_sha256',
        'aggregation_implementation_sha256',
        'positive_label_multilabel_reduction',
    ],
    'scripts/revision/31_generate_external_comparator_predictions.py': [
        'validate_checkpoint_set',
        '--fold-cache-dir',
        'Verified final external comparator artifacts were not reusable',
    ],
    'scripts/revision/32_paired_external_comparators.py': [
        'Paired external bootstrap requires at least two groups',
        'probabilities must be in [0,1]',
    ],
    'scripts/revision/33_group_safe_score_calibration.py': [
        '--primary-fraction',
        'pre_specified_before_test_metric_evaluation',
        'independent_target_groups_from_adaptation_pool',
    ],
    'scripts/revision/34_extract_external_representations.py': [
        'checkpoint_source_contract',
        'validate_checkpoint_files_against_oof_run_manifest',
        'frozen_encoder_external_record_representation_v2_source_bound',
        'validate_source_provenance',
        'current runner/canonical contract',
    ],
    'scripts/revision/35_true_fewshot_head_adaptation.py': [
        'embedding_manifest_path',
        'Embedding manifest is stale or incomplete',
        'independent_target_groups_from_adaptation_pool',
        'REPRESENTATION_PROTOCOL_VERSION = 2',
        'pre_specified_before_test_metric_evaluation',
        'primary_endpoint_rows',
    ],
    'scripts/revision/50_refresh_in_domain_paired_contracts.py': [
        'IN_DOMAIN_PAIRED_REFRESH_CAPABILITY',
        'canonical_freeze_sha256',
        'regenerated_cpu',
    ],
    'scripts/revision/51_ptbxl_adaptation_analysis_lock.py': [
        'PTBXL_ADAPTATION_LOCK_CAPABILITY',
        'post_initial_result_review',
        'evaluate_fold10_only_after_configuration_lock_validation',
    ],
    'scripts/revision/52_ptbxl_fold_protocol_audit.py': [
        'PTBXL_FOLD_PROTOCOL_AUDIT_CAPABILITY',
        'patient-group percentile bootstrap',
        'sensitivity_exclude_unsupported_only',
    ],
}
REVISION_REQUIRED_FILES = [
    'docs/revision_plan/georgia_label_mapping_review_20260703.csv',
]


def _literal_module_assignments(path):
    try:
        tree = _compat_ast.parse(path.read_text(encoding='utf-8'), filename=str(path))
    except Exception as exc:
        raise RuntimeError(f'Could not parse capability source {path}: {exc}') from exc
    assignments = {}
    for node in tree.body:
        if isinstance(node, _compat_ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], _compat_ast.Name):
            try:
                assignments[node.targets[0].id] = _compat_ast.literal_eval(node.value)
            except (ValueError, TypeError):
                continue
        elif isinstance(node, _compat_ast.AnnAssign) and isinstance(node.target, _compat_ast.Name):
            try:
                assignments[node.target.id] = _compat_ast.literal_eval(node.value)
            except (ValueError, TypeError):
                continue
    return assignments


compatibility_failures = []
for rel_path, expected in REVISION_CAPABILITY_REQUIREMENTS.items():
    path = REPO_DIR / rel_path
    if not path.is_file() or path.stat().st_size == 0:
        compatibility_failures.append(f'{rel_path}: missing or empty')
        continue
    observed = _literal_module_assignments(path)
    mismatches = {
        key: {'expected': value, 'observed': observed.get(key)}
        for key, value in expected.items()
        if observed.get(key) != value
    }
    if mismatches:
        compatibility_failures.append(f'{rel_path}: capability mismatch {mismatches}')

for rel_path, required_tokens in REVISION_TOKEN_REQUIREMENTS.items():
    path = REPO_DIR / rel_path
    if not path.is_file() or path.stat().st_size == 0:
        compatibility_failures.append(f'{rel_path}: missing or empty')
        continue
    source_text = path.read_text(encoding='utf-8', errors='replace')
    missing_tokens = [token for token in required_tokens if token not in source_text]
    if missing_tokens:
        compatibility_failures.append(f'{rel_path}: missing contract tokens {missing_tokens}')

for rel_path in REVISION_REQUIRED_FILES:
    path = REPO_DIR / rel_path
    if not path.is_file() or path.stat().st_size == 0:
        compatibility_failures.append(f'{rel_path}: missing or empty')

if compatibility_failures:
    authority = (CODE_AUTHORITY or {}).get('git_commit', 'unknown')
    raise RuntimeError(
        f'Notebook 02 is incompatible with pinned authority {authority}: '
        + ' ; '.join(compatibility_failures)
        + '. Do not hotfix a detached authority checkout from mutable GitHub main. '
        'Commit and review the required change, then rotate authority through Notebook 00.'
    )
print('Revision capability preflight: OK')
# END FORENSIC NOTEBOOK 02 CAPABILITY PREFLIGHT

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('chapman_zip:', chapman_zip, '| size=', chapman_zip.stat().st_size)
print('model_dir  :', model_dir)
print('branch    :', BRANCH)

from configs.config import PATHS

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache_pattern': 'rocket_raw_N44186_C12_L5000_K10000_S42*.npz',
    'hrv36_cache_pattern': 'hrv36_N44186_C12_L5000*.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_pca_models',
    'fold_hydra_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, target in cache_status.items():
    if isinstance(target, str):
        matches = sorted(DRIVE_ROOT.glob(target))
        print(f'  {name}: count={len(matches)} pattern={target}')
        for match in matches[:5]:
            print(f'    - {match.name} size={match.stat().st_size}')
    else:
        path = target
        if path.is_dir():
            npz_count = len(list(path.glob('*.npz')))
            joblib_count = len(list(path.glob('*.joblib')))
            print(f'  {name}: exists=True npz_count={npz_count} joblib_count={joblib_count} path={path}')
        else:
            print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

# Expected feature-cache names are fingerprinted from the cleaned cache subject order.
# If these files exist, train/evaluation will load them instead of regenerating MiniRocket/HRV.
def inspect_expected_feature_caches():
    import numpy as np
    from src.provenance import record_order_fingerprint

    clean_cache_path = Path(PATHS['data_cache'])
    if not clean_cache_path.is_file():
        print('Feature cache preflight: clean cache is missing, so exact MiniRocket/HRV fingerprint cannot be derived yet.')
        return

    with np.load(clean_cache_path, allow_pickle=True) as payload:
        subjects = payload['subjects']
        stored_fingerprint = (
            str(payload['record_order_fingerprint'].item())
            if 'record_order_fingerprint' in payload.files
            else None
        )
    computed_fingerprint = record_order_fingerprint(subjects)
    if stored_fingerprint and stored_fingerprint != computed_fingerprint:
        raise RuntimeError(
            'Clean cache fingerprint mismatch: '
            f'stored={stored_fingerprint} computed={computed_fingerprint}'
        )
    fingerprint = stored_fingerprint or computed_fingerprint
    n_records = len(subjects)
    cache_dir = Path(PATHS['cache_dir'])
    expected_feature_caches = {
        'expected RAW MiniRocket cache': cache_dir / f'rocket_raw_N{n_records}_C12_L5000_K10000_S42_R{fingerprint}.npz',
        'expected HRV36 cache': cache_dir / f'hrv36_N{n_records}_C12_L5000_R{fingerprint}.npz',
    }
    print('Feature cache fingerprint:', fingerprint)
    all_present = True
    for name, path in expected_feature_caches.items():
        exists = path.is_file()
        all_present = all_present and exists
        size = path.stat().st_size if exists else None
        size_gib = f'{size / (1024 ** 3):.2f} GiB' if size is not None else '-'
        print(f'{name:32s}: exists={exists} size={size_gib} path={path}')
    if all_present:
        print('Feature cache preflight: MiniRocket and HRV36 caches are present and should be reused.')
    else:
        print('Feature cache preflight: at least one feature cache is missing; that feature will be regenerated once and then cached.')

inspect_expected_feature_caches()

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        log_relative = log_path.resolve().relative_to((REPO_DIR / 'reports' / 'revision').resolve())
    except ValueError:
        log_relative = Path('logs') / log_path.name
    durable_log_path = MIRROR_REVISION_ROOT / log_relative
    durable_log_path.parent.mkdir(parents=True, exist_ok=True)

    from contextlib import ExitStack
    with ExitStack() as stack:
        log_file = stack.enter_context(log_path.open('a', encoding='utf-8'))
        durable_log_file = stack.enter_context(durable_log_path.open('a', encoding='utf-8'))
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
            durable_log_file.write(line)
            durable_log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    print(f'Durable command log: {durable_log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


def require_gpu_inference_runtime(stage, require_mamba=True):
    """Fail before expensive preprocessing when pending inference lacks its runtime."""
    import importlib.util

    modules = ['wfdb', 'torch']
    if require_mamba:
        modules.extend(['mamba_ssm', 'causal_conv1d'])
    missing = [name for name in modules if importlib.util.find_spec(name) is None]
    if missing:
        raise RuntimeError(
            f'{stage} requires GPU inference, but runtime modules are missing: {missing}. '
            'Run Setup, Install Base Dependencies, and Install Model Dependencies in this runtime.'
        )
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            f'{stage} requires pending model inference. Select an A100 High-RAM runtime and rerun from Setup. '
            'Already published canonical artifacts remain reusable.'
        )
    name = torch.cuda.get_device_name(0)
    memory_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'{stage} inference runtime: {name} ({memory_gib:.1f} GiB)')
    return name

# Forensic run-history wrapper. The legacy helper writes live output while this
# wrapper gives every invocation a unique, durable stage/run_id log and retains
# the requested stable path as the latest-run convenience copy.
FORENSIC_RUN_HISTORY_CAPABILITY = 'stage_run_id_v1'
_forensic_base_run = run

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    import os as _forensic_os
    import shutil as _forensic_shutil
    import subprocess as _forensic_subprocess
    import time as _forensic_time
    import uuid as _forensic_uuid
    from datetime import datetime as _forensic_datetime, timezone as _forensic_timezone
    from pathlib import Path as _ForensicPath

    run_cwd = _ForensicPath(cwd) if cwd else _ForensicPath.cwd()
    if log_path is None:
        stable_log = run_cwd / 'reports' / 'revision' / 'logs' / 'notebook_command_latest.log'
    else:
        stable_log = _ForensicPath(log_path)
        if not stable_log.is_absolute():
            stable_log = run_cwd / stable_log
    stable_log.parent.mkdir(parents=True, exist_ok=True)
    stage = stable_log.stem
    run_id = _forensic_datetime.now(_forensic_timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ') + '-' + _forensic_uuid.uuid4().hex[:10]
    history_log = stable_log.parent / 'history' / stage / f'{run_id}.log'
    history_log.parent.mkdir(parents=True, exist_ok=True)

    canonical_root = globals().get('MIRROR_REVISION_ROOT')
    if canonical_root is None and 'DRIVE_ROOT' in globals():
        canonical_root = _ForensicPath(DRIVE_ROOT) / 'revision_artifacts' / 'reports' / 'revision'
    canonical_history = None
    if canonical_root is not None:
        canonical_root = _ForensicPath(canonical_root)
        canonical_history = canonical_root / 'logs' / 'history' / stage / f'{run_id}.log'
        canonical_history.parent.mkdir(parents=True, exist_ok=True)

    started = _forensic_datetime.now(_forensic_timezone.utc).isoformat()
    header = f'run_id={run_id}\nstage={stage}\nstarted_utc={started}\ncommand={cmd}\n--- output ---\n'
    history_log.write_text(header, encoding='utf-8')
    if canonical_history is not None:
        canonical_history.write_text(header, encoding='utf-8')

    return_code = -1
    caught = None
    completed = None
    process = None
    try:
        process = _forensic_subprocess.Popen(
            cmd,
            shell=isinstance(cmd, str),
            cwd=str(run_cwd),
            stdout=_forensic_subprocess.PIPE,
            stderr=_forensic_subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        with history_log.open('a', encoding='utf-8') as local_handle:
            canonical_handle = (
                canonical_history.open('a', encoding='utf-8')
                if canonical_history is not None
                else None
            )
            try:
                for line in process.stdout or ():
                    print(line, end='', flush=True)
                    local_handle.write(line)
                    local_handle.flush()
                    if canonical_handle is not None:
                        canonical_handle.write(line)
                        canonical_handle.flush()
                return_code = int(process.wait())
                local_handle.flush()
                _forensic_os.fsync(local_handle.fileno())
                if canonical_handle is not None:
                    canonical_handle.flush()
                    _forensic_os.fsync(canonical_handle.fileno())
            finally:
                if canonical_handle is not None:
                    canonical_handle.close()
        completed = _forensic_subprocess.CompletedProcess(cmd, return_code)
    except BaseException as exc:
        caught = exc
        return_code = int(getattr(exc, 'returncode', -1))
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except Exception:
                process.kill()
                process.wait()
    finally:
        footer = (
            '\n--- end ---\n'
            f'ended_utc={_forensic_datetime.now(_forensic_timezone.utc).isoformat()}\n'
            f'return_code={return_code}\n'
        )
        with history_log.open('a', encoding='utf-8') as handle:
            handle.write(footer)
            handle.flush()
            _forensic_os.fsync(handle.fileno())
        if canonical_history is not None:
            # The underlying helper streams to this same canonical history path
            # when supported; append the footer or refresh from the local copy.
            try:
                _forensic_shutil.copy2(history_log, canonical_history)
            except Exception as exc:
                print('WARNING: durable history refresh failed:', exc)
        try:
            _forensic_shutil.copy2(history_log, stable_log)
            if canonical_root is not None:
                try:
                    revision_base = (_ForensicPath(globals().get('REPO_DIR', run_cwd)) / 'reports' / 'revision').resolve()
                    relative = stable_log.resolve().relative_to(revision_base)
                except (ValueError, TypeError):
                    relative = _ForensicPath('logs') / stable_log.name
                canonical_stable = canonical_root / relative
                canonical_stable.parent.mkdir(parents=True, exist_ok=True)
                _forensic_shutil.copy2(history_log, canonical_stable)
        except Exception as exc:
            print('WARNING: latest log refresh failed:', exc)
    print('Run history log:', history_log)
    if canonical_history is not None:
        print('Durable run history log:', canonical_history)
    if caught is not None:
        raise caught
    if check and return_code:
        raise _forensic_subprocess.CalledProcessError(return_code, cmd)
    return completed


## Install Base Dependencies

This notebook can run directly after opening in Colab. These packages cover data loading, feature extraction, metrics, and artifact checks.

In [ ]:
BASE_INSTALLER_CAPABILITY = 'ecg_ramba_base_installer_v1'
BASE_INSTALLER_SCHEMA_VERSION = 1
INSTALL_BASE_DEPS = True
RESTART_RUNTIME_AFTER_BASE_DEP_CHANGE = True
REPAIR_BROKEN_NUMERIC_STACK = True

if INSTALL_BASE_DEPS:
    import importlib
    import importlib.metadata as metadata
    import json
    import os
    import subprocess
    import sys
    import time

    numeric_packages = [
        "numpy>=2.0,<2.1",
        "scipy>=1.14.1,<2.0",
        "scikit-learn>=1.2,<1.9",
    ]
    support_packages = [
        "pandas==2.2.2",
        "requests==2.32.4",
        "pillow>=8,<12",
        "threadpoolctl",
        "tqdm",
        "wfdb==4.1.2",
        "joblib",
        "matplotlib",
        "seaborn",
        "packaging",
        "neurokit2",
        "iterative-stratification",
        "thop",
        "einops",
        "ninja",
    ]
    numeric_dists = {
        "numpy": "numpy",
        "scipy": "scipy",
        "scikit-learn": "scikit-learn",
    }
    runtime_dir = DRIVE_ROOT / "colab_runtime"
    runtime_dir.mkdir(parents=True, exist_ok=True)

    def dist_version(dist_name):
        try:
            return metadata.version(dist_name)
        except metadata.PackageNotFoundError:
            return None

    def numeric_versions():
        return {name: dist_version(dist) for name, dist in numeric_dists.items()}

    def pip_install(extra_args, packages):
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                *extra_args,
                *packages,
            ],
            check=True,
        )

    def restart_runtime(reason):
        print("")
        print("=" * 80)
        print("Intentional Colab runtime restart")
        print("=" * 80)
        print(reason)
        print("After Colab reconnects, run this notebook again from the first cell.")
        print("Base dependency state is stored under:", runtime_dir)
        print("=" * 80)
        sys.stdout.flush()
        time.sleep(8)
        try:
            from google.colab import runtime
            runtime.restart_runtime()
        except Exception:
            os.kill(os.getpid(), 9)

    def sanity_import_numeric_stack():
        import numpy as np
        from numpy._core import strings as _np_strings  # catches mixed NumPy installs on Colab
        import scipy
        import sklearn
        import wfdb
        return {
            "numpy": np.__version__,
            "scipy": scipy.__version__,
            "sklearn": sklearn.__version__,
            "wfdb": wfdb.__version__,
        }

    print("Python:", sys.version)
    print("Installing/updating ECG-RAMBA runtime dependencies without downgrading Colab numpy/scipy.")
    before_versions = numeric_versions()
    pip_install(["--upgrade", "--upgrade-strategy", "only-if-needed"], numeric_packages + support_packages)
    after_versions = numeric_versions()
    changed_versions = {
        name: {"before": before_versions.get(name), "after": after_versions.get(name)}
        for name in numeric_dists
        if before_versions.get(name) != after_versions.get(name)
    }
    if changed_versions and RESTART_RUNTIME_AFTER_BASE_DEP_CHANGE:
        manifest = {
            "reason": "Base numeric dependencies changed; restart required before importing sklearn/NumPy extensions.",
            "changed_versions": changed_versions,
            "python": sys.version,
        }
        manifest_path = runtime_dir / f"base_deps_restart_py{sys.version_info.major}{sys.version_info.minor}.json"
        manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        restart_runtime("Base numeric dependencies changed: " + json.dumps(changed_versions))

    try:
        versions = sanity_import_numeric_stack()
    except Exception as exc:
        print("Numeric stack import sanity check failed:", repr(exc))
        if REPAIR_BROKEN_NUMERIC_STACK:
            print("Force-reinstalling NumPy/SciPy/scikit-learn once, then restarting runtime.")
            pip_install(["--force-reinstall", "--no-cache-dir"], numeric_packages)
            repair_manifest = {
                "reason": "Numeric stack import failed after pip install; repaired with force-reinstall.",
                "error": repr(exc),
                "versions_before_repair": after_versions,
                "versions_after_repair": numeric_versions(),
                "python": sys.version,
            }
            repair_path = runtime_dir / f"numeric_stack_repair_py{sys.version_info.major}{sys.version_info.minor}.json"
            repair_path.write_text(json.dumps(repair_manifest, indent=2), encoding="utf-8")
            restart_runtime("NumPy/SciPy/scikit-learn were force-reinstalled after an import failure.")
        raise

    for name, version in versions.items():
        print(f"{name:8s}: {version}")

    check = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if check.returncode:
        print("pip check reported remaining non-ECG Colab package conflicts:")
        print(check.stdout)
        print("Continue only if the ECG-RAMBA imports above succeeded; unused Colab packages can still report conflicts.")
    else:
        print("pip check: OK")
else:
    print('Skipping base dependency install.')


## Install Model Dependencies

`auto` installs the CUDA/Mamba runtime only when a GPU is attached. CPU reuse/gate sessions skip this cell safely; pending inference stages still fail later with a stage-specific A100 requirement.

In [ ]:
MAMBA_INSTALLER_CAPABILITY = 'ecg_ramba_mamba_installer_v1'
MAMBA_INSTALLER_SCHEMA_VERSION = 1
INSTALL_MODEL_DEPS = 'auto'
DOWNLOAD_MAMBA_WHEELS_IF_MISSING = True
AUTO_PIN_TORCH_FOR_MAMBA = True
RESTART_RUNTIME_AFTER_TORCH_PIN = True
UNINSTALL_TORCH_COMPANION_PACKAGES = True

import importlib.util as _model_dep_importlib_util
_model_dep_torch_available = _model_dep_importlib_util.find_spec('torch') is not None
if _model_dep_torch_available:
    import torch as _model_dep_torch
    MODEL_DEPS_GPU_AVAILABLE = bool(_model_dep_torch.cuda.is_available() and _model_dep_torch.version.cuda)
else:
    MODEL_DEPS_GPU_AVAILABLE = False
MODEL_DEPS_SHOULD_RUN = INSTALL_MODEL_DEPS is True or (
    str(INSTALL_MODEL_DEPS).strip().lower() == 'auto' and MODEL_DEPS_GPU_AVAILABLE
)
if INSTALL_MODEL_DEPS is True and not MODEL_DEPS_GPU_AVAILABLE:
    raise RuntimeError('INSTALL_MODEL_DEPS=True was forced, but this runtime has no CUDA-enabled PyTorch.')
print('Model dependency policy:', INSTALL_MODEL_DEPS, '| CUDA available=', MODEL_DEPS_GPU_AVAILABLE, '| should_run=', MODEL_DEPS_SHOULD_RUN)

if MODEL_DEPS_SHOULD_RUN:
    import hashlib
    import json
    import os
    import re
    import subprocess
    import sys
    import time
    import urllib.request

    import torch

    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_major_minor = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cuda_version = torch.version.cuda or ""
    cuda_tag = f"cu{cuda_version.split('.')[0]}" if cuda_version else None
    abi_tag = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"

    wheel_cache_dir = DRIVE_ROOT / f"mamba_wheels_py{sys.version_info.major}{sys.version_info.minor}"
    legacy_dirs = [
        Path("/content/drive/MyDrive") / wheel_cache_dir.name,
        Path("/content/drive/MyDrive/mamba_wheels_py312"),
        DRIVE_ROOT / "mamba_wheels_py312",
    ]
    explicit_wheel_dirs = []
    for p in [wheel_cache_dir, *legacy_dirs]:
        if p not in explicit_wheel_dirs:
            explicit_wheel_dirs.append(p)

    print("Mamba wheel environment")
    print("  Python tag :", py_tag)
    print("  torch      :", torch.__version__)
    print("  CUDA       :", cuda_version or "CPU")
    print("  CUDA tag   :", cuda_tag or "none")
    print("  CXX11 ABI  :", abi_tag)
    print("  cache dir  :", wheel_cache_dir)

    if cuda_tag is None:
        raise RuntimeError("CUDA-enabled PyTorch is required. In Colab, select a GPU runtime before running this notebook.")

    release_cache = {}
    torch_companion_packages = ["torchvision", "torchaudio", "torchtext"]

    def restart_runtime_after_pin() -> None:
        print("")
        print("=" * 80)
        print("Intentional Colab runtime restart")
        print("=" * 80)
        print("Torch was changed to a Mamba-compatible version.")
        print("After Colab reconnects, run this notebook again from the first cell.")
        print("The downloaded/pinned artifacts are stored on Drive and will be reused.")
        print("=" * 80)
        sys.stdout.flush()
        time.sleep(8)
        try:
            from google.colab import runtime
            runtime.restart_runtime()
        except Exception:
            os.kill(os.getpid(), 9)

    def sha256_file(path: Path) -> str:
        h = hashlib.sha256()
        with path.open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()

    def wheel_role(path_or_name) -> str | None:
        name = Path(path_or_name).name
        if name.startswith("causal_conv1d-"):
            return "causal"
        if name.startswith("mamba_ssm-"):
            return "mamba"
        return None

    def compatible_wheel(path_or_name, role: str, torch_minor: str | None = None) -> bool:
        name = Path(path_or_name).name
        if wheel_role(name) != role:
            return False
        torch_minor = torch_minor or torch_major_minor
        required = [
            f"{py_tag}-{py_tag}",
            "linux_x86_64.whl",
            f"torch{torch_minor}",
            f"cxx11abi{abi_tag}",
        ]
        if cuda_tag:
            required.append(cuda_tag)
        return all(token in name for token in required)

    def list_wheels():
        wheels = []
        for wheel_dir in explicit_wheel_dirs:
            if wheel_dir.exists():
                wheels.extend(sorted(wheel_dir.glob("*.whl")))
        return sorted(set(wheels))

    def select_cached_wheels():
        wheels = list_wheels()
        if wheels:
            print(f"Found {len(wheels)} wheel file(s) in configured Drive cache folders.")
            for p in wheels[:60]:
                flag = "compatible" if compatible_wheel(p, wheel_role(p) or "") else "present"
                print(f"  - {p.name} [{flag}]")
            if len(wheels) > 60:
                print(f"  ... {len(wheels) - 60} more wheel(s)")
        causal = [p for p in wheels if compatible_wheel(p, "causal")]
        mamba = [p for p in wheels if compatible_wheel(p, "mamba")]
        return (causal[0] if causal else None), (mamba[0] if mamba else None)

    def github_latest_release(repo: str) -> dict:
        if repo in release_cache:
            return release_cache[repo]
        url = f"https://api.github.com/repos/{repo}/releases/latest"
        req = urllib.request.Request(url, headers={"User-Agent": "ECG-RAMBA-Colab"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            release_cache[repo] = json.loads(resp.read().decode("utf-8"))
        return release_cache[repo]

    def asset_torch_minor(asset_name: str, role: str) -> str | None:
        if wheel_role(asset_name) != role:
            return None
        required = [
            f"{py_tag}-{py_tag}",
            "linux_x86_64.whl",
            f"cxx11abi{abi_tag}",
        ]
        if cuda_tag:
            required.append(cuda_tag)
        if not all(token in asset_name for token in required):
            return None
        m = re.search(r"torch([0-9]+\.[0-9]+)", asset_name)
        return m.group(1) if m else None

    def release_supported_torch_minors(repo: str, role: str) -> set[str]:
        rel = github_latest_release(repo)
        minors = {
            minor for asset in rel.get("assets", [])
            if (minor := asset_torch_minor(asset["name"], role)) is not None
        }
        return minors

    def torch_minor_key(minor: str) -> tuple[int, int]:
        major, minor_part = minor.split(".", 1)
        return int(major), int(minor_part)

    def choose_supported_torch_minor() -> tuple[str | None, list[str]]:
        causal_minors = release_supported_torch_minors("Dao-AILab/causal-conv1d", "causal")
        mamba_minors = release_supported_torch_minors("state-spaces/mamba", "mamba")
        common = sorted(causal_minors & mamba_minors, key=torch_minor_key)
        common_stable = [m for m in common if m.startswith("2.")]
        if torch_major_minor in common:
            return torch_major_minor, common

        requested = os.environ.get("ECG_RAMBA_TORCH_MINOR")
        if requested:
            if requested not in common:
                raise RuntimeError(f"ECG_RAMBA_TORCH_MINOR={requested} is not supported by both Mamba wheel releases: {common}")
            return requested, common

        lower_or_equal = [
            m for m in common_stable
            if torch_minor_key(m) <= torch_minor_key(torch_major_minor)
        ]
        if lower_or_equal:
            return lower_or_equal[-1], common
        if common_stable:
            return common_stable[-1], common
        return None, common

    def version_tuple(version: str) -> tuple[int, int, int]:
        parts = version.split("+", 1)[0].split(".")
        nums = [int(p) if p.isdigit() else 0 for p in parts[:3]]
        while len(nums) < 3:
            nums.append(0)
        return tuple(nums)

    def latest_pypi_torch_version(target_minor: str) -> str:
        url = "https://pypi.org/pypi/torch/json"
        req = urllib.request.Request(url, headers={"User-Agent": "ECG-RAMBA-Colab"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
        versions = [
            version for version, files in payload.get("releases", {}).items()
            if version.startswith(target_minor + ".")
            and re.match(r"^[0-9]+\.[0-9]+\.[0-9]+$", version)
            and files
        ]
        if not versions:
            raise RuntimeError(f"No stable torch release found on PyPI for torch{target_minor}.")
        return sorted(versions, key=version_tuple)[-1]

    def pin_torch_if_needed() -> None:
        target_minor, supported = choose_supported_torch_minor()
        print("Mamba-supported torch minors:", supported)
        if target_minor == torch_major_minor:
            return
        if target_minor is None:
            raise RuntimeError(
                f"No common Mamba wheel support found for {py_tag}, {cuda_tag}, cxx11abi{abi_tag}. "
                "Use a different Colab runtime or build wheels manually."
            )
        if not AUTO_PIN_TORCH_FOR_MAMBA:
            raise RuntimeError(
                f"Current torch{torch_major_minor} has no matching Mamba wheels. "
                f"Set AUTO_PIN_TORCH_FOR_MAMBA=True or install torch{target_minor} manually."
            )

        target_version = latest_pypi_torch_version(target_minor)
        pin_manifest = {
            "reason": "Current Colab torch minor has no matching Mamba release wheels.",
            "current_torch_version": torch.__version__,
            "current_torch_minor": torch_major_minor,
            "target_torch_version": target_version,
            "target_torch_minor": target_minor,
            "supported_torch_minors": supported,
            "python_tag": py_tag,
            "cuda_tag": cuda_tag,
            "cxx11abi": abi_tag,
            "uninstalled_torch_companion_packages": (
                torch_companion_packages if UNINSTALL_TORCH_COMPANION_PACKAGES else []
            ),
        }
        wheel_cache_dir.mkdir(parents=True, exist_ok=True)
        pin_path = wheel_cache_dir / "torch_runtime_pin.json"
        pin_path.write_text(json.dumps(pin_manifest, indent=2), encoding="utf-8")
        print(f"Current torch{torch_major_minor} has no compatible Mamba wheels.")
        print(f"Installing torch=={target_version} so Mamba wheels can use torch{target_minor}.")
        print(f"Wrote torch pin manifest: {pin_path}")
        if UNINSTALL_TORCH_COMPANION_PACKAGES:
            print("Removing Torch companion packages that can pin a different torch version:")
            print("  " + ", ".join(torch_companion_packages))
            subprocess.run(
                [sys.executable, "-m", "pip", "uninstall", "-y", *torch_companion_packages],
                check=False,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", f"torch=={target_version}"],
            check=True,
        )
        if RESTART_RUNTIME_AFTER_TORCH_PIN:
            restart_runtime_after_pin()
        raise SystemExit("Restart runtime and rerun the notebook from the top.")

    def select_release_asset(repo: str, role: str) -> dict:
        rel = github_latest_release(repo)
        assets = rel.get("assets", [])
        matches = [a for a in assets if compatible_wheel(a["name"], role)]
        if matches:
            print(f"Selected {role} wheel from {repo} {rel.get('tag_name')}: {matches[0]['name']}")
            return matches[0]

        candidates = [
            a["name"] for a in assets
            if wheel_role(a["name"]) == role and py_tag in a["name"] and "linux_x86_64.whl" in a["name"]
        ]
        print(f"No compatible {role} wheel found in latest release {repo} {rel.get('tag_name')}.")
        print("Closest cp/linux candidates:")
        for name in candidates[:30]:
            print("  -", name)
        raise RuntimeError(
            f"Missing compatible {role} wheel for {py_tag}, torch{torch_major_minor}, "
            f"{cuda_tag}, cxx11abi{abi_tag}. Runtime and wheel tags must match."
        )

    def download_asset(asset: dict, out_dir: Path) -> Path:
        out_dir.mkdir(parents=True, exist_ok=True)
        name = asset["name"]
        out_path = out_dir / name
        if out_path.exists() and out_path.stat().st_size == int(asset.get("size", 0)):
            print(f"Reusing cached download: {out_path}")
            return out_path

        tmp_path = out_path.with_suffix(out_path.suffix + ".partial")
        print(f"Downloading {name}")
        print(f"  -> {out_path}")
        req = urllib.request.Request(asset["browser_download_url"], headers={"User-Agent": "ECG-RAMBA-Colab"})
        with urllib.request.urlopen(req, timeout=900) as resp, tmp_path.open("wb") as f:
            while True:
                chunk = resp.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)
        os.replace(tmp_path, out_path)
        return out_path

    pin_torch_if_needed()

    causal_wheel, mamba_wheel = select_cached_wheels()
    downloaded_assets = []

    if (causal_wheel is None or mamba_wheel is None) and DOWNLOAD_MAMBA_WHEELS_IF_MISSING:
        print("Compatible cached Mamba wheels are missing; downloading release wheels to Drive.")
        if causal_wheel is None:
            asset = select_release_asset("Dao-AILab/causal-conv1d", "causal")
            causal_wheel = download_asset(asset, wheel_cache_dir)
            downloaded_assets.append(asset)
        if mamba_wheel is None:
            asset = select_release_asset("state-spaces/mamba", "mamba")
            mamba_wheel = download_asset(asset, wheel_cache_dir)
            downloaded_assets.append(asset)

    if causal_wheel is None or mamba_wheel is None:
        raise FileNotFoundError(
            "Missing compatible causal_conv1d*.whl and/or mamba_ssm*.whl. "
            "Set DOWNLOAD_MAMBA_WHEELS_IF_MISSING=True or place matching wheels in the Drive cache."
        )

    manifest = {
        "python_tag": py_tag,
        "torch_version": torch.__version__,
        "torch_major_minor": torch_major_minor,
        "cuda_version": cuda_version,
        "cuda_tag": cuda_tag,
        "cxx11abi": abi_tag,
        "wheel_cache_dir": str(wheel_cache_dir),
        "causal_wheel": {
            "path": str(causal_wheel),
            "name": causal_wheel.name,
            "sha256": sha256_file(causal_wheel),
            "bytes": causal_wheel.stat().st_size,
        },
        "mamba_wheel": {
            "path": str(mamba_wheel),
            "name": mamba_wheel.name,
            "sha256": sha256_file(mamba_wheel),
            "bytes": mamba_wheel.stat().st_size,
        },
        "downloaded_assets": [
            {
                "name": a.get("name"),
                "url": a.get("browser_download_url"),
                "size": a.get("size"),
                "digest": a.get("digest"),
            }
            for a in downloaded_assets
        ],
    }
    manifest_path = wheel_cache_dir / "wheel_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Wrote wheel manifest: {manifest_path}")

    print("Installing Mamba wheels from Drive cache")
    print("  causal-conv1d:", causal_wheel.name)
    print("  mamba-ssm    :", mamba_wheel.name)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(causal_wheel), "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(mamba_wheel), "-q"], check=True)

    import mamba_ssm
    print("mamba_ssm import OK:", mamba_ssm.__file__)
else:
    if str(INSTALL_MODEL_DEPS).strip().lower() == 'auto' and not MODEL_DEPS_GPU_AVAILABLE:
        print('CPU reuse-only runtime: skipping CUDA/Mamba installation. Protocol gates, paired statistics, and mirror operations may continue.')
    else:
        print('Mamba installation disabled explicitly. Pending ECG-RAMBA inference will require a prepared CUDA runtime.')


## Restore And Verify Stable Drive Artifacts

Restore only files declared by the Drive mirror manifest. Every source and destination is verified by SHA256 before it can be reused.

In [ ]:
stable_mirror = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
OOF_RESTORE_PATHS = [
    'predictions/oof_final_ema_predictions.npz',
    'predictions/oof_final_ema_slice_predictions.npz',
    'metrics/oof_final_ema_prediction_summary.json',
    'tables/oof_final_ema_class_summary.csv',
    'manifests/oof_final_ema_prediction_run_manifest.json',
    'manifests/oof_final_ema_freeze_manifest.json',
]
if stable_mirror.exists():
    include_args = ' '.join(f'--include-path "{path}"' for path in OOF_RESTORE_PATHS)
    restore_result = run(
        f'python -u scripts/revision/artifact_mirror.py restore '
        f'--mirror-root "{stable_mirror}" --replace-mismatched {include_args}',
        check=False,
        log_path='reports/revision/logs/oof_mirror_targeted_restore.log',
    )
    if restore_result.returncode:
        print('Targeted verified OOF restore unavailable. Fold-cache/OOF validation will decide whether inference is required.')
else:
    print('Stable mirror does not exist yet:', stable_mirror)


## Prediction Contract

Every downstream metric script expects NPZ files with `y_true` and `y_prob`, both shaped `(N, C)`. Store prediction files under `reports/revision/predictions/`.

In [ ]:
from pathlib import Path
import numpy as np

pred_dir = Path('reports/revision/predictions')
pred_dir.mkdir(parents=True, exist_ok=True)

for path in sorted(pred_dir.glob('*.npz')):
    data = np.load(path, allow_pickle=True)
    keys = sorted(data.files)
    print(path, keys)
    if {'y_true', 'y_prob'}.issubset(keys):
        print('  y_true:', data['y_true'].shape, 'y_prob:', data['y_prob'].shape)


## Repair Legacy OOF Aggregation

This step reuses saved slice probabilities only when all five folds and all valid records are present. Otherwise it leaves artifacts untouched.

In [ ]:
RUN_STANDALONE_LEGACY_REAGGREGATION = False

if RUN_STANDALONE_LEGACY_REAGGREGATION:
    run(
        'python -u scripts/revision/02_reaggregate_oof.py --expected-folds 5 --q 3 --if-possible',
        check=False,
        log_path='reports/revision/logs/oof_reaggregate.log',
    )
else:
    print('Standalone repair skipped; Generate OOF Predictions performs this repair automatically.')


## Generate OOF Predictions

In [ ]:
RUN_OOF_EXPORT = True
FORCE_RERUN_OOF = False
RESUME_FOLD_CACHE = True
FORCE_RERUN_FOLDS = False
REQUIRE_HIGH_RAM = True
MIN_SYSTEM_RAM_GB = 24
BATCH_SIZE = 256
NUM_WORKERS = 2
DEBUG_LIMIT_RECORDS = 0
CANONICAL_OOF_KIND = 'final_ema'
CANONICAL_OOF_STEM = 'oof_final_ema'
CANONICAL_FREEZE_MANIFEST = Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json')
OOF_FOLD_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'predictions' / 'folds'
OOF_FOLD_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# CPU-only OOF provenance restore. A missing group sidecar is a metadata-contract
# gap, not evidence that model inference must be repeated.
CANONICAL_GROUP_SIDECAR = Path('reports/revision/manifests/oof_final_ema_group_sidecar.npz')
OOF_CORE_ARTIFACTS = [
    Path(f'reports/revision/predictions/{CANONICAL_OOF_STEM}_predictions.npz'),
    Path(f'reports/revision/predictions/{CANONICAL_OOF_STEM}_slice_predictions.npz'),
    Path(f'reports/revision/metrics/{CANONICAL_OOF_STEM}_prediction_summary.json'),
    Path(f'reports/revision/tables/{CANONICAL_OOF_STEM}_class_summary.csv'),
    Path(f'reports/revision/manifests/{CANONICAL_OOF_STEM}_prediction_run_manifest.json'),
]
OOF_CONTRACT_RESTORE_PATHS = [
    *OOF_CORE_ARTIFACTS,
    CANONICAL_FREEZE_MANIFEST,
    CANONICAL_GROUP_SIDECAR,
    Path('reports/revision/logs/oof_final_ema_generate_predictions.log'),
]
oof_restore_args = ' '.join(
    f'--include-path "{path.relative_to(Path("reports/revision")).as_posix()}"'
    for path in OOF_CONTRACT_RESTORE_PATHS
)
run(
    f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{stable_mirror}" '
    f'--replace-mismatched {oof_restore_args}',
    log_path='reports/revision/logs/oof_contract_targeted_restore.log',
)


def ensure_oof_group_sidecar():
    record_path = OOF_CORE_ARTIFACTS[0]
    if not record_path.is_file() or record_path.stat().st_size == 0:
        print('OOF group sidecar build deferred because canonical OOF predictions are absent.')
        return False
    run(
        'python -u scripts/revision/49_build_oof_group_sidecar.py '
        f'--oof-predictions "{record_path}" --source-archive "{chapman_zip}" '
        f'--out "{CANONICAL_GROUP_SIDECAR}" --expected-records 44186 --reuse-existing',
        log_path='reports/revision/logs/oof_group_sidecar.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size '
        f'--source-conflict-policy source '
        f'--include-path "manifests/oof_final_ema_group_sidecar.npz" '
        f'--mirror-root "{stable_mirror}"',
        log_path='reports/revision/logs/oof_group_sidecar_immediate_mirror_publish.log',
    )
    return True


ensure_oof_group_sidecar()

import os
import shutil
import zipfile

if not chapman_zip.is_file() or not zipfile.is_zipfile(chapman_zip):
    raise FileNotFoundError(f'Valid Chapman ZIP is required before OOF generation: {chapman_zip}')
required_model_files = [model_dir / 'folds.pkl']
for fold in range(1, 6):
    required_model_files.append(model_dir / f'fold{fold}_{CANONICAL_OOF_KIND}.pt')
missing_model_files = [str(path) for path in required_model_files if not path.is_file()]
if missing_model_files:
    raise FileNotFoundError(
        'Canonical OOF requires post-fix explicit EMA checkpoints. '
        'Run the retraining notebook/command first. Missing: ' + '; '.join(missing_model_files)
    )
print('Canonical OOF input preflight: Chapman ZIP, folds.pkl, and five fixed-epoch final_ema checkpoints are available.')

page_size = os.sysconf('SC_PAGE_SIZE')
physical_pages = os.sysconf('SC_PHYS_PAGES')
system_ram_gb = page_size * physical_pages / (1024 ** 3)
disk = shutil.disk_usage('/content')
print(f'System RAM : {system_ram_gb:.1f} GiB')
print(f'Local disk : {disk.free / (1024 ** 3):.1f} GiB free')

try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    gpu_ram_gb = (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        if torch.cuda.is_available() else 0.0
    )
except Exception:
    gpu_name, gpu_ram_gb = 'unknown', 0.0
print(f'GPU        : {gpu_name} ({gpu_ram_gb:.1f} GiB)')

freeze_command = (
    'python -u scripts/revision/06_freeze_oof.py '
    f'--record-file reports/revision/predictions/{CANONICAL_OOF_STEM}_predictions.npz '
    f'--slice-file reports/revision/predictions/{CANONICAL_OOF_STEM}_slice_predictions.npz '
    f'--summary-file reports/revision/metrics/{CANONICAL_OOF_STEM}_prediction_summary.json '
    f'--class-table reports/revision/tables/{CANONICAL_OOF_STEM}_class_summary.csv '
    f'--run-manifest reports/revision/manifests/{CANONICAL_OOF_STEM}_prediction_run_manifest.json '
    f'--freeze-manifest {CANONICAL_FREEZE_MANIFEST} '
    '--expected-records 44186 --expected-folds 5 --q 3 --expected-checkpoint-kind final_ema --manuscript-ready-strict --group-sidecar reports/revision/manifests/oof_final_ema_group_sidecar.npz'
)
freeze_check_command = freeze_command + ' --check-existing-freeze'


def freeze_oof(label):
    print(f'Validating canonical final_ema OOF contract ({label})...')
    result = run(
        freeze_check_command,
        check=False,
        log_path='reports/revision/logs/oof_final_ema_freeze_validation.log',
    )
    if result.returncode == 0:
        print('final_ema OOF passed checksum, five-fold, 44,186-record, Q=3, and checkpoint validation.')
        return True
    print('final_ema OOF strict contract is not ready; attempting CPU metadata refresh before any inference decision.')
    return False


oof_ready = False if FORCE_RERUN_OOF else freeze_oof('existing artifacts')
oof_core_available = all(path.is_file() and path.stat().st_size > 0 for path in OOF_CORE_ARTIFACTS)
if not FORCE_RERUN_OOF and not oof_ready and oof_core_available and CANONICAL_GROUP_SIDECAR.is_file():
    print('Refreshing the strict OOF freeze metadata on CPU; existing predictions remain unchanged.')
    freeze_refresh_command = freeze_command + ' --metadata-refresh-from-existing-oof'
    refresh_result = run(
        freeze_refresh_command,
        check=False,
        log_path='reports/revision/logs/oof_final_ema_freeze_refresh.log',
    )
    if refresh_result.returncode == 0:
        freeze_publish_args = ' '.join([
            '--include-path "manifests/oof_final_ema_group_sidecar.npz"',
            '--include-path "manifests/oof_final_ema_freeze_manifest.json"',
        ])
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size '
            f'--source-conflict-policy source {freeze_publish_args} --mirror-root "{stable_mirror}"',
            log_path='reports/revision/logs/oof_freeze_refresh_immediate_mirror_publish.log',
        )
        oof_ready = freeze_oof('after CPU metadata refresh')
    else:
        print('CPU freeze metadata refresh failed; model inference will not be started automatically while complete OOF artifacts exist.')
command = (
    'python -u scripts/revision/01_generate_predictions.py '
    f'--dataset oof --checkpoint-kind {CANONICAL_OOF_KIND} --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} '
    f'--fold-cache-dir "{OOF_FOLD_CACHE_DIR}"'
)
if not RESUME_FOLD_CACHE:
    command += ' --no-resume-fold-cache'
if FORCE_RERUN_FOLDS:
    command += ' --force-rerun-folds'
if DEBUG_LIMIT_RECORDS:
    command += f' --limit-records {DEBUG_LIMIT_RECORDS}'

oof_inference_required = bool(FORCE_RERUN_OOF or not oof_core_available)
if RUN_OOF_EXPORT and oof_inference_required:
    require_gpu_inference_runtime('Canonical final_ema OOF export')
    if REQUIRE_HIGH_RAM and system_ram_gb < MIN_SYSTEM_RAM_GB:
        raise RuntimeError(
            f'Insufficient system RAM: {system_ram_gb:.1f} GiB available, '
            f'but the full final_ema OOF pipeline requires at least {MIN_SYSTEM_RAM_GB} GiB. '
            'Exit code 137 on a standard T4 runtime is a host-RAM kill, not a CUDA batch-size issue. '
            'Select a High-RAM runtime/A100, then rerun from the setup cell. '
            'Do not reduce numerical precision for manuscript predictions.'
        )
    run(command, log_path='reports/revision/logs/oof_final_ema_generate_predictions.log')
    ensure_oof_group_sidecar()
    run(freeze_command, log_path='reports/revision/logs/oof_final_ema_freeze_validation.log')
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{stable_mirror}"',
        log_path='reports/revision/logs/oof_final_ema_immediate_mirror_publish.log',
    )
elif oof_ready and not FORCE_RERUN_OOF:
    print('Skipping final_ema OOF inference because the frozen artifact contract passed.')
elif oof_core_available:
    raise RuntimeError(
        'Complete OOF prediction artifacts are present, but their strict provenance/freeze contract failed. '
        'GPU inference was intentionally not started. Review oof_final_ema_freeze_refresh.log and repair the '
        'specific sidecar, split-membership, or freeze metadata blocker; set FORCE_RERUN_OOF=True only when the '
        'prediction artifact itself is proven invalid.'
    )
else:
    print(f'OOF export disabled. Set RUN_OOF_EXPORT=True to execute: {command}')


## Verify OOF Prediction Outputs

In [ ]:
run(
    'python -u scripts/revision/06_freeze_oof.py '
    '--record-file reports/revision/predictions/oof_final_ema_predictions.npz '
    '--slice-file reports/revision/predictions/oof_final_ema_slice_predictions.npz '
    '--summary-file reports/revision/metrics/oof_final_ema_prediction_summary.json '
    '--class-table reports/revision/tables/oof_final_ema_class_summary.csv '
    '--run-manifest reports/revision/manifests/oof_final_ema_prediction_run_manifest.json '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-records 44186 --expected-folds 5 --q 3 --expected-checkpoint-kind final_ema --manuscript-ready-strict --group-sidecar reports/revision/manifests/oof_final_ema_group_sidecar.npz --check-existing-freeze',
    log_path='reports/revision/logs/oof_final_ema_verify_contract.log',
)

expected = [
    Path('reports/revision/predictions/oof_final_ema_predictions.npz'),
    Path('reports/revision/predictions/oof_final_ema_slice_predictions.npz'),
    Path('reports/revision/metrics/oof_final_ema_prediction_summary.json'),
    Path('reports/revision/tables/oof_final_ema_class_summary.csv'),
    Path('reports/revision/manifests/oof_final_ema_prediction_run_manifest.json'),
    Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json'),
]

for path in expected:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

fold_cache_dir = globals().get(
    'OOF_FOLD_CACHE_DIR',
    DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'predictions' / 'folds',
)
fold_cache_files = sorted(fold_cache_dir.glob('oof_fold*_final_ema_*.npz')) if fold_cache_dir.exists() else []
print('\nFinal EMA fold cache files:', len(fold_cache_files))
for path in fold_cache_files:
    print(' -', path, path.stat().st_size, 'bytes')

pred_path = expected[0]
if pred_path.exists():
    data = np.load(pred_path, allow_pickle=True)
    print('keys:', sorted(data.files))
    print('y_true:', data['y_true'].shape)
    print('y_prob:', data['y_prob'].shape)
    print('classes:', list(data['class_names'])[:5], '...', len(data['class_names']))
    for key in [
        'dataset',
        'protocol',
        'config_hash',
        'git_commit',
        'checkpoint_kind',
        'batch_size',
        'aggregation_method',
        'aggregation_q',
        'aggregation_implementation',
        'cache_schema_version',
        'source_config_hash',
        'evaluation_config_hash',
    ]:
        if key in data.files:
            value = data[key]
            print(f'{key}:', value.item() if value.ndim == 0 else value)

summary_path = Path('reports/revision/metrics/oof_final_ema_prediction_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('\nSummary keys:', sorted(summary.keys()))
    print('metrics:', summary.get('metrics'))
    print('slice_count_summary:', summary.get('slice_count_summary'))

manifest_path = Path('reports/revision/manifests/oof_final_ema_prediction_run_manifest.json')
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('\nManifest runtime:', manifest.get('runtime', {}).get('torch'))
    print('Manifest outputs:')
    for name, info in manifest.get('outputs', {}).items():
        print(' -', name, info.get('path'), info.get('size_bytes'), info.get('sha256', '')[:12])

freeze_path = Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json')
if freeze_path.exists():
    freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
    print('\nFreeze status:', freeze.get('status'))
    print('Manuscript ready:', freeze.get('manuscript_ready'))
    print('Validated records:', freeze.get('validated_records'))
    print('Checkpoint kind:', freeze.get('checkpoint_kind'))
    print('Fold counts:', freeze.get('fold_counts'))
    print('Checkpoint match:', freeze.get('checkpoint_fingerprints_match'))
    if freeze.get('checkpoint_kind') != 'final_ema':
        raise RuntimeError('Canonical freeze manifest must report checkpoint_kind=final_ema')


## Diagnostic Final Checkpoint OOF Audit

This optional diagnostic run evaluates `fold*_final_raw.pt` separately from the manuscript `final_ema` OOF artifacts. It writes `oof_final_raw_*` artifacts and must not replace frozen `oof_final_ema_*` files.


In [ ]:
RUN_FINAL_OOF_AUDIT = False
FORCE_RERUN_FINAL_OOF = False
FINAL_BATCH_SIZE = BATCH_SIZE
FINAL_NUM_WORKERS = NUM_WORKERS
FINAL_N_BOOT = 1000

final_expected = {
    'prediction': Path('reports/revision/predictions/oof_final_raw_predictions.npz'),
    'slice_prediction': Path('reports/revision/predictions/oof_final_raw_slice_predictions.npz'),
    'summary': Path('reports/revision/metrics/oof_final_raw_prediction_summary.json'),
    'class_summary': Path('reports/revision/tables/oof_final_raw_class_summary.csv'),
    'run_manifest': Path('reports/revision/manifests/oof_final_raw_prediction_run_manifest.json'),
    'calibration_ci': Path('reports/revision/metrics/calibration_ci_oof_final_raw_predictions.json'),
}

final_command = (
    'python -u scripts/revision/01_generate_predictions.py '
    f'--dataset oof --checkpoint-kind final_raw --batch-size {FINAL_BATCH_SIZE} --num-workers {FINAL_NUM_WORKERS}'
)
if not RESUME_FOLD_CACHE:
    final_command += ' --no-resume-fold-cache'
if FORCE_RERUN_FOLDS:
    final_command += ' --force-rerun-folds'
if DEBUG_LIMIT_RECORDS:
    final_command += f' --limit-records {DEBUG_LIMIT_RECORDS}'

if not RUN_FINAL_OOF_AUDIT:
    print('Final raw checkpoint audit is disabled. It is diagnostic only and must not replace final_ema OOF.')
    print('Planned command:', final_command)
else:
    final_model_files = [model_dir / f'fold{fold}_final_raw.pt' for fold in range(1, 6)]
    missing_final_models = [str(path) for path in final_model_files if not path.is_file()]
    if missing_final_models:
        raise FileNotFoundError('Final raw audit requires all fold*_final_raw.pt files: ' + '; '.join(missing_final_models))

    final_prediction_ready = all(final_expected[key].exists() for key in ['prediction', 'summary', 'class_summary', 'run_manifest'])
    if FORCE_RERUN_FINAL_OOF or not final_prediction_ready:
        if REQUIRE_HIGH_RAM and system_ram_gb < MIN_SYSTEM_RAM_GB:
            raise RuntimeError(
                f'Insufficient system RAM: {system_ram_gb:.1f} GiB available, '
                f'but the final raw checkpoint OOF audit requires at least {MIN_SYSTEM_RAM_GB} GiB.'
            )
        run(final_command, log_path='reports/revision/logs/oof_final_raw_generate_predictions.log')
    else:
        print('Skipping final raw checkpoint OOF inference because oof_final_raw artifacts already exist.')

    pred_path = final_expected['prediction']
    if not pred_path.exists():
        raise FileNotFoundError(f'Final raw OOF prediction file was not created: {pred_path}')

    with np.load(pred_path, allow_pickle=True) as data:
        print('Final raw prediction keys:', sorted(data.files))
        print('Final raw y_true:', data['y_true'].shape)
        print('Final raw y_prob:', data['y_prob'].shape)
        print('Final raw protocol:', data['protocol'].item() if data['protocol'].ndim == 0 else data['protocol'])
        checkpoint_kind = data['checkpoint_kind'].item() if data['checkpoint_kind'].ndim == 0 else str(data['checkpoint_kind'])
        print('Final raw checkpoint_kind:', checkpoint_kind)
        if tuple(data['y_prob'].shape) != (44186, 27):
            raise RuntimeError(f'Unexpected final raw y_prob shape: {data["y_prob"].shape}')
        if tuple(data['y_true'].shape) != (44186, 27):
            raise RuntimeError(f'Unexpected final raw y_true shape: {data["y_true"].shape}')
        if checkpoint_kind != 'final_raw':
            raise RuntimeError(f'Final raw audit loaded the wrong checkpoint kind: {checkpoint_kind}')
        fold_id = data['fold_id']
        if np.any(fold_id < 0):
            raise RuntimeError(f'Final raw OOF has missing predictions: {int(np.sum(fold_id < 0))}')
        fold_counts = {int(fold): int(np.sum(fold_id == fold)) for fold in sorted(np.unique(fold_id))}
        print('Final raw fold counts:', fold_counts)
        if set(fold_counts) != {1, 2, 3, 4, 5} or sum(fold_counts.values()) != 44186:
            raise RuntimeError(f'Unexpected final raw fold coverage: {fold_counts}')

    calibration_cmd = (
        'python -u scripts/revision/04_calibration_ci.py '
        '--predictions "reports/revision/predictions/oof_final_raw_predictions.npz" '
        '--out "reports/revision/metrics/calibration_ci_oof_final_raw_predictions.json" '
        f'--threshold 0.5 --n-bins 15 --n-boot {FINAL_N_BOOT}'
    )
    if FORCE_RERUN_FINAL_OOF or not final_expected['calibration_ci'].exists():
        run(calibration_cmd, log_path='reports/revision/logs/oof_final_raw_calibration_ci.log')
    else:
        print('Skipping final raw calibration/CI because diagnostic output already exists.')

    for name, path in final_expected.items():
        print(f'{name:16s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

    canonical_calibration = Path('reports/revision/metrics/calibration_ci_oof_final_ema_predictions.json')
    final_calibration = final_expected['calibration_ci']
    if canonical_calibration.exists() and final_calibration.exists():
        best_payload = json.loads(canonical_calibration.read_text(encoding='utf-8'))
        final_payload = json.loads(final_calibration.read_text(encoding='utf-8'))
        rows = []
        for label, payload in [('final_ema/canonical', best_payload), ('final_raw/diagnostic', final_payload)]:
            metrics = payload.get('metrics', {})
            calibration = payload.get('calibration', {})
            rows.append({
                'run': label,
                'f1_macro': metrics.get('f1_macro'),
                'pr_auc_macro': metrics.get('pr_auc_macro'),
                'roc_auc_macro': metrics.get('roc_auc_macro'),
                'ece_macro': calibration.get('ece_macro'),
                'brier_macro': calibration.get('brier_macro'),
            })
        print('\nCanonical final_ema vs final_raw diagnostic comparison:')
        for row in rows:
            print(row)


## Experimental External Prediction Commands

Full reviewer-evidence plan: keep Chapman frozen OOF as the primary in-domain evidence and evaluate PTB-XL, Georgia, and CPSC2021 only as separate protocol-gated mapped tasks. PTB-XL and Georgia are record-level tasks; CPSC2021 is an annotation-aligned AF/AFL window task and is never pooled with them. External source exports remain `manuscript_ready=false` until their dataset-specific gates pass.


In [ ]:
# Expanded external-evidence run: reuse existing artifacts and regenerate missing external predictions.
from pathlib import Path
import json
import os
import shutil
import subprocess
import numpy as np
import pandas as pd

DRIVE_ROOT = Path(globals().get('DRIVE_ROOT', Path(os.environ.get('ECG_RAMBA_DRIVE_ROOT', '/content/drive/MyDrive/ECG-Ramba'))))
DRIVE_MOUNT = Path('/content/drive')

def _direct_drive_root_ready(root):
    try:
        return root.is_dir() and any(root.iterdir())
    except Exception:
        return False

if not _direct_drive_root_ready(DRIVE_ROOT):
    try:
        from google.colab import drive
        try:
            drive.mount('/content/drive')
        except Exception as exc:
            print(f'Direct external cell Drive mount initial attempt failed or was stale: {exc}')
            print('Retrying Drive mount with force_remount=True ...')
            drive.mount('/content/drive', force_remount=True)
    except Exception as exc:
        print(f'Drive mount skipped or unavailable in direct external cell: {exc}')
if not _direct_drive_root_ready(DRIVE_ROOT):
    visible_mount = sorted(path.name for path in DRIVE_MOUNT.iterdir()) if DRIVE_MOUNT.exists() else []
    visible_my_drive = sorted(path.name for path in (DRIVE_MOUNT / 'MyDrive').iterdir()) if (DRIVE_MOUNT / 'MyDrive').exists() else []
    raise RuntimeError(
        f'Drive root is still not visible at {DRIVE_ROOT}. '
        f'Visible under {DRIVE_MOUNT}: {visible_mount}; visible under MyDrive: {visible_my_drive}. '
        'Reconnect Drive with force_remount=True before external artifact restore/export.'
    )
REPO_DIR = globals().get('REPO_DIR', Path(os.environ.get('ECG_RAMBA_REPO_DIR', '/content/ECG-RAMBA')))
if not Path('scripts/revision/03_generate_external_predictions.py').exists() and Path(REPO_DIR).exists():
    os.chdir(REPO_DIR)
    print('Changed working directory for direct external rerun:', Path.cwd())

if 'run' not in globals():
    def run(cmd, check=True, log_path=None, tail_lines=25, cwd=None):
        print(f'$ {cmd}', flush=True)
        run_cwd = Path(cwd) if cwd else Path.cwd()
        local_log = Path(log_path) if log_path else None
        if local_log is not None and not local_log.is_absolute():
            local_log = run_cwd / local_log
        mirror_root = Path(globals().get('MIRROR_REVISION_ROOT', DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'))
        durable_log = None
        if local_log is not None:
            try:
                rel = local_log.resolve().relative_to((run_cwd / 'reports' / 'revision').resolve())
            except ValueError:
                rel = Path('logs') / local_log.name
            durable_log = mirror_root / rel
            local_log.parent.mkdir(parents=True, exist_ok=True)
            durable_log.parent.mkdir(parents=True, exist_ok=True)
        proc = subprocess.Popen(cmd, shell=True, cwd=str(run_cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
        assert proc.stdout is not None
        tail = []
        local_handle = local_log.open('a', encoding='utf-8') if local_log else None
        durable_handle = durable_log.open('a', encoding='utf-8') if durable_log else None
        durable_failed = False
        try:
            for line in proc.stdout:
                print(line, end='', flush=True)
                tail = (tail + [line.rstrip('\n')])[-tail_lines:]
                if local_handle:
                    local_handle.write(line); local_handle.flush()
                if durable_handle:
                    try:
                        durable_handle.write(line); durable_handle.flush()
                    except OSError as exc:
                        if not durable_failed:
                            print(f'WARNING: durable Drive log became unavailable: {exc}', flush=True)
                        durable_failed = True
                        durable_handle.close(); durable_handle = None
            return_code = proc.wait()
        finally:
            if local_handle: local_handle.close()
            if durable_handle: durable_handle.close()
        if local_log: print(f'Command log: {local_log}')
        if durable_log and not durable_failed: print(f'Durable command log: {durable_log}')
        if check and return_code != 0:
            print(f'Command failed with exit code {return_code}. Last {tail_lines} log lines:')
            for line in tail:
                print(line)
            raise subprocess.CalledProcessError(return_code, cmd)
        return subprocess.CompletedProcess(cmd, return_code)

# PTB-XL is already manuscript-gated; Georgia/CPSC2021 will run only when their prediction artifacts are missing.
# Allowed values for BUILD_FOLD_PCA and RUN_*_EXPORT: True, False, or 'auto'.
BUILD_FOLD_PCA = 'auto'
RUN_PTBXL_EXPORT = 'auto'
RUN_GEORGIA_EXPORT = 'auto'
RUN_CPSC2021_EXPORT = 'auto'
EXTERNAL_BATCH_SIZE = 128
EXTERNAL_LIMIT_RECORDS = 0
ALLOW_EXTERNAL_EXPORT_FAILURES = True
GEORGIA_MAPPING_REVIEW = Path('docs/revision_plan/georgia_label_mapping_review_20260703.csv')
GEORGIA_CODE_INVENTORY_OUT = Path('reports/revision/tables/table_georgia_snomed_code_inventory.csv')
CPSC_ANNOTATION_AUDIT_OUT = Path('reports/revision/tables/table_cpsc2021_annotation_audit.csv')
FOLD_PCA_MANIFEST = Path('reports/revision/manifests/fold_pca_manifest.json')

# Direct-rerun convenience: restore external/gate/few-shot artifacts from Drive mirror
# before deciding whether expensive export/model inference is needed. This avoids
# rerunning the earlier full mirror-restore cell after a Colab disconnect.
AUTO_RESTORE_EXTERNAL_ARTIFACTS = True
EXTERNAL_RESTORE_DATASETS = ['ptbxl', 'georgia', 'cpsc2021']


def _report_rel(path):
    path = Path(path)
    prefix = Path('reports/revision')
    try:
        return path.relative_to(prefix)
    except ValueError:
        return None


def _restore_report_artifact(path, source_roots, remove_unpublished_active=False):
    from scripts.revision.common import sha256_file

    path = Path(path)
    rel = _report_rel(path)
    if rel is None:
        return {'path': str(path), 'status': 'not_report_artifact', 'source': '', 'size': None}
    for root in source_roots:
        root = Path(root)
        manifest_path = root / 'manifests' / 'mirror_manifest.json'
        if not manifest_path.exists() or manifest_path.stat().st_size == 0:
            continue
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        rows = {row.get('relative_path'): row for row in manifest.get('artifacts', [])}
        row = rows.get(rel.as_posix())
        source = root / rel
        if row is None or not source.exists() or source.stat().st_size == 0:
            continue
        expected_size = int(row.get('size_bytes', -1))
        expected_sha = str(row.get('sha256') or '')
        if expected_size >= 0 and source.stat().st_size != expected_size:
            raise RuntimeError(f'Canonical mirror size mismatch for {rel}')
        if len(expected_sha) != 64 or sha256_file(source) != expected_sha:
            raise RuntimeError(f'Canonical mirror checksum mismatch for {rel}')
        if path.exists() and path.stat().st_size > 0 and sha256_file(path) == expected_sha:
            return {'path': str(path), 'status': 'reused_active', 'source': str(source), 'size': path.stat().st_size}
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, path)
        if sha256_file(path) != expected_sha:
            raise RuntimeError(f'Checksum mismatch after restoring {rel}')
        return {'path': str(path), 'status': 'restored', 'source': str(source), 'size': path.stat().st_size}
    if path.exists() and path.stat().st_size > 0:
        if remove_unpublished_active:
            size = path.stat().st_size
            path.unlink()
            return {'path': str(path), 'status': 'removed_unpublished_active', 'source': '', 'size': size}
        raise RuntimeError(
            f'Active artifact is not authenticated by the canonical mirror manifest: {path}. '
            'Publish the producing notebook before reuse.'
        )
    return {'path': str(path), 'status': 'missing_in_sources', 'source': '', 'size': None}


def _external_artifact_paths_for_restore(datasets):
    paths = [FOLD_PCA_MANIFEST]
    external_base = Path('reports/revision/experimental/external')
    for dataset in datasets:
        paths.extend([
            external_base / dataset / f'{dataset}_full_predictions.npz',
            external_base / dataset / f'{dataset}_full_slice_predictions.npz',
            external_base / dataset / f'{dataset}_full_prediction_summary.json',
            external_base / dataset / f'{dataset}_full_prediction_run_manifest.json',
            external_base / dataset / f'{dataset}_full_class_summary.csv',
            Path(f'reports/revision/metrics/external_{dataset}_protocol_gate.json'),
            Path(f'reports/revision/tables/table_external_{dataset}_label_mapping.csv'),
            Path(f'reports/revision/tables/table_external_{dataset}_metrics.csv'),
            Path(f'reports/revision/manifests/external_{dataset}_protocol_gate_manifest.json'),
            Path(f'reports/revision/metrics/fewshot_{dataset}_summary.csv'),
            Path(f'reports/revision/tables/table_fewshot_{dataset}.csv'),
            Path(f'reports/revision/metrics/fewshot_{dataset}_bootstrap.json'),
            Path(f'reports/revision/manifests/fewshot_{dataset}_splits.npz'),
            Path(f'reports/revision/manifests/fewshot_{dataset}_run_manifest.json'),
        ])
    paths.extend([
        Path('reports/revision/metrics/external_protocol_gate_summary.csv'),
        Path('reports/revision/tables/table_georgia_snomed_code_inventory.csv'),
        Path('reports/revision/tables/table_cpsc2021_annotation_audit.csv'),
    ])
    deduped = []
    seen = set()
    for path in paths:
        key = str(path)
        if key not in seen:
            deduped.append(path)
            seen.add(key)
    return deduped


if AUTO_RESTORE_EXTERNAL_ARTIFACTS:
    mirror_root = globals().get('stable_mirror', DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision')
    restore_source_roots = [Path(mirror_root)]
    restore_source_roots = [root for root in restore_source_roots if root.exists()]
    print('External targeted restore source roots:', [str(root) for root in restore_source_roots])
    restore_rows = [
        _restore_report_artifact(path, restore_source_roots)
        for path in _external_artifact_paths_for_restore(EXTERNAL_RESTORE_DATASETS)
    ]
    restore_df = pd.DataFrame(restore_rows)
    restored_count = int((restore_df['status'] == 'restored').sum()) if not restore_df.empty else 0
    missing_count = int((restore_df['status'] == 'missing_in_sources').sum()) if not restore_df.empty else 0
    print(f'External targeted restore: restored={restored_count} missing_in_sources={missing_count}')
    display(restore_df[restore_df['status'].isin(['restored', 'missing_in_sources'])])

required_external_cli_tokens = [
    '--georgia-mapping-review',
    '--georgia-code-inventory-out',
    '--cpsc-annotation-audit-out',
    'preprocess_georgia_mat_record',
    'from scipy.io import loadmat',
]
external_script = Path('scripts/revision/03_generate_external_predictions.py')
external_script_source = external_script.read_text(encoding='utf-8') if external_script.exists() else ''
missing_external_cli_tokens = [token for token in required_external_cli_tokens if token not in external_script_source]
if missing_external_cli_tokens:
    raise RuntimeError(
        'External exporter script is stale and cannot run Georgia MAT/CPSC2021 mapped/audited export. '
        'Pull latest main / rerun Notebook 00 setup, or run from a repo containing the updated '
        'scripts/revision/03_generate_external_predictions.py. '
        f'Missing CLI tokens: {missing_external_cli_tokens}'
    )


def first_existing_drive_file(names):
    candidates = [DRIVE_ROOT / name for name in names]
    found = next((path for path in candidates if path.is_file()), None)
    return found, candidates


def external_required_artifacts(dataset):
    base = Path('reports/revision/experimental/external')
    return [
        base / dataset / f'{dataset}_full_predictions.npz',
        base / dataset / f'{dataset}_full_slice_predictions.npz',
        base / dataset / f'{dataset}_full_prediction_summary.json',
        base / dataset / f'{dataset}_full_prediction_run_manifest.json',
        base / dataset / f'{dataset}_full_class_summary.csv',
    ]


external_reuse_diagnostics = {}


def external_prediction_ready(dataset):
    from scripts.revision.external_reuse_contract import validate_external_prediction_reuse

    mirror_root = Path(globals().get(
        'MIRROR_REVISION_ROOT',
        DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision',
    ))
    result = validate_external_prediction_reuse(
        dataset,
        revision_root=Path('reports/revision'),
        archive_path=external_archives.get(dataset),
        exporter_path=Path('scripts/revision/03_generate_external_predictions.py'),
        oof_path=Path('reports/revision/predictions/oof_final_ema_predictions.npz'),
        freeze_path=Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json'),
        archive_hash_cache_dir=mirror_root / 'manifests' / 'external_archive_hash_cache',
        threshold=0.5,
        q=3.0,
    )
    reasons = list(result.get('reasons', []))
    external_reuse_diagnostics[dataset] = reasons
    print(f'{dataset} external source-bound reuse contract ready={result.get("ready", False)}')
    for path in external_required_artifacts(dataset):
        print(f'  {path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
    if reasons:
        print('  reuse rejected:', '; '.join(reasons))
    diagnostics = result.get('diagnostics', {})
    if diagnostics.get('q3_reconstruction_max_abs') is not None:
        print('  Q=3 reconstruction max_abs:', diagnostics['q3_reconstruction_max_abs'])
    return bool(result.get('ready', False))

def resolve_auto_flag(value, ready):
    if isinstance(value, str) and value.strip().lower() == 'auto':
        return not ready
    return bool(value)


external_archive_specs = {
    'ptbxl': ['PTB-XL.zip', 'ptb-xl.zip', 'ptbxl.zip'],
    'georgia': ['Georgia.zip'],
    'cpsc2021': ['cpsc2021.zip', 'CPSC2021.zip'],
}
external_archives = {}
for dataset, names in external_archive_specs.items():
    found, candidates = first_existing_drive_file(names)
    external_archives[dataset] = found
    print(f'{dataset} archive:', found if found else 'MISSING', '| checked=', [str(p) for p in candidates])

external_flag_config = {
    'ptbxl': RUN_PTBXL_EXPORT,
    'georgia': RUN_GEORGIA_EXPORT,
    'cpsc2021': RUN_CPSC2021_EXPORT,
}
external_ready = {dataset: external_prediction_ready(dataset) for dataset in external_flag_config}
external_jobs = [
    (dataset, resolve_auto_flag(flag, external_ready[dataset]))
    for dataset, flag in external_flag_config.items()
]
print('Resolved external export jobs:', external_jobs)

# Preflight summary: use this table to decide whether a GPU export is needed after a runtime reset.
preflight_rows = []
for dataset, export_enabled in external_jobs:
    gate_json = Path(f'reports/revision/metrics/external_{dataset}_protocol_gate.json')
    fewshot_summary = Path(f'reports/revision/metrics/fewshot_{dataset}_summary.csv')
    gate_status = ''
    gate_passed = None
    if gate_json.exists() and gate_json.stat().st_size > 0:
        try:
            gate_payload = json.loads(gate_json.read_text(encoding='utf-8'))
            gate_status = gate_payload.get('status', '')
            gate_passed = gate_payload.get('protocol_gate_passed')
        except Exception as exc:
            gate_status = f'unreadable: {exc}'
    preflight_rows.append({
        'dataset': dataset,
        'archive_exists': external_archives.get(dataset) is not None,
        'external_artifacts_ready': bool(external_ready[dataset]),
        'export_will_run': bool(export_enabled),
        'gate_json_exists': gate_json.exists(),
        'gate_status': gate_status,
        'gate_passed': gate_passed,
        'fewshot_summary_exists': fewshot_summary.exists(),
    })
preflight_df = pd.DataFrame(preflight_rows)
display(preflight_df)

need_external_export = any(enabled for _, enabled in external_jobs)
fold_pca_ready = FOLD_PCA_MANIFEST.exists() and FOLD_PCA_MANIFEST.stat().st_size > 0
if need_external_export:
    print('External model inference is needed for at least one dataset. Use A100 High-RAM if available.')
else:
    print('All enabled external prediction artifacts are reusable. CPU runtime is enough for gate/few-shot/publish cells.')
build_fold_pca = resolve_auto_flag(BUILD_FOLD_PCA, fold_pca_ready) and need_external_export
print('Fold PCA manifest:', FOLD_PCA_MANIFEST, 'exists=', FOLD_PCA_MANIFEST.exists(), 'size=', FOLD_PCA_MANIFEST.stat().st_size if FOLD_PCA_MANIFEST.exists() else None)
print('Resolved BUILD_FOLD_PCA:', build_fold_pca)
if need_external_export:
    pending_external_datasets = [dataset for dataset, enabled in external_jobs if enabled]
    print('Pending external re-export datasets:', pending_external_datasets)
    for dataset in pending_external_datasets:
        print(f'  {dataset} reuse rejection:', external_reuse_diagnostics.get(dataset, ['unknown']))
    require_gpu_inference_runtime('External Full-model re-export for ' + ','.join(pending_external_datasets))

if build_fold_pca:
    run(
        'python -u scripts/revision/08_build_fold_pca.py --checkpoint-kind final_ema',
        log_path='reports/revision/logs/build_fold_pca.log',
    )
elif need_external_export and not fold_pca_ready:
    raise FileNotFoundError(
        f'External export is required, but {FOLD_PCA_MANIFEST} is missing. '
        "Set BUILD_FOLD_PCA='auto' or True, or restore the fold PCA manifest from the mirror."
    )
else:
    print('Fold PCA build skipped because no missing enabled external export requires it, or manifest is already present.')

external_export_failures = []
external_export_ran = False
for dataset, enabled in external_jobs:
    if enabled and external_archives.get(dataset) is None:
        message = f'{dataset} export enabled but archive is missing under {DRIVE_ROOT}'
        external_export_failures.append({'dataset': dataset, 'reason': message, 'log_path': None})
        if not ALLOW_EXTERNAL_EXPORT_FAILURES:
            raise FileNotFoundError(message)
        print('External export skipped:', message)
        continue
    command = (
        'python -u scripts/revision/03_generate_external_predictions.py '
        f'--dataset {dataset} --checkpoint-kind final_ema --batch-size {EXTERNAL_BATCH_SIZE} '
        '--allow-experimental'
    )
    if dataset == 'georgia':
        command += f' --georgia-mapping-review "{GEORGIA_MAPPING_REVIEW}" --georgia-code-inventory-out "{GEORGIA_CODE_INVENTORY_OUT}"'
    if dataset == 'cpsc2021':
        command += f' --cpsc-annotation-audit-out "{CPSC_ANNOTATION_AUDIT_OUT}"'
    if EXTERNAL_LIMIT_RECORDS:
        command += f' --limit-records {EXTERNAL_LIMIT_RECORDS}'
    if enabled:
        log_path = f'reports/revision/logs/{dataset}_generate_predictions.log'
        try:
            run(command, log_path=log_path)
            external_export_ran = True
            mirror_root = globals().get('stable_mirror', DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision')
            external_publish_paths = [
                *external_required_artifacts(dataset),
                Path('reports/revision/experimental/external/external_summary_experimental.csv'),
            ]
            if dataset == 'georgia':
                external_publish_paths.append(GEORGIA_CODE_INVENTORY_OUT)
            elif dataset == 'cpsc2021':
                external_publish_paths.append(CPSC_ANNOTATION_AUDIT_OUT)
            external_publish_args = ' '.join(
                f'--include-path "{path.relative_to(Path("reports/revision")).as_posix()}"'
                for path in external_publish_paths
            )
            run(
                f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size '
                f'--source-conflict-policy source {external_publish_args} --mirror-root "{mirror_root}"',
                log_path=f'reports/revision/logs/{dataset}_external_export_mirror_publish.log',
            )
        except subprocess.CalledProcessError as exc:
            external_export_failures.append(
                {
                    'dataset': dataset,
                    'reason': f'export command failed with exit code {exc.returncode}',
                    'log_path': log_path,
                }
            )
            print(f'External export failed for {dataset}; log={log_path}')
            print('This dataset will remain blocked/deferred unless a later successful export creates gate-ready artifacts.')
            if not ALLOW_EXTERNAL_EXPORT_FAILURES:
                raise
    else:
        print(f'{dataset}: export not needed. Set RUN_{dataset.upper()}_EXPORT=True to force rerun.')

if external_export_failures:
    print('External export failures recorded:')
    for failure in external_export_failures:
        print(f"- {failure['dataset']}: {failure['reason']} | log={failure['log_path']}")
    print('Continue to the protocol gate with EXTERNAL_GATE_STRICT=False if you want failures recorded as blocked/deferred.')


if external_export_ran:
    print('Successful external exports were published with exact source-path selection; no broad mirror overwrite is needed.')
else:
    print('No external export command ran; mirror publish for external exports skipped.')

# Storage handoff barrier: do not disconnect the GPU until every gate input is both
# present locally and authenticated by the canonical Drive mirror manifest.
REQUIRE_EXTERNAL_CACHE_HANDOFF = True
external_handoff_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
external_handoff_manifest = external_handoff_root / 'manifests' / 'mirror_manifest.json'
external_handoff_issues = []
for dataset in EXTERNAL_RESTORE_DATASETS:
    if not external_prediction_ready(dataset):
        reasons = external_reuse_diagnostics.get(dataset, ['unknown semantic contract failure'])
        external_handoff_issues.append(f'{dataset} semantic contract invalid: ' + '; '.join(reasons))
if not external_handoff_manifest.exists() or external_handoff_manifest.stat().st_size == 0:
    external_handoff_issues.append(f'missing canonical manifest: {external_handoff_manifest}')
else:
    from scripts.revision.common import sha256_file
    handoff_payload = json.loads(external_handoff_manifest.read_text(encoding='utf-8'))
    handoff_rows = {row.get('relative_path'): row for row in handoff_payload.get('artifacts', [])}
    for local_path in [path for dataset in EXTERNAL_RESTORE_DATASETS for path in external_required_artifacts(dataset)]:
        rel = local_path.relative_to(Path('reports/revision')).as_posix()
        canonical_path = external_handoff_root / rel
        row = handoff_rows.get(rel)
        if not local_path.exists() or local_path.stat().st_size == 0:
            external_handoff_issues.append(f'active missing: {rel}')
            continue
        if row is None or not canonical_path.exists() or canonical_path.stat().st_size == 0:
            external_handoff_issues.append(f'canonical/manifest missing: {rel}')
            continue
        local_sha = sha256_file(local_path)
        canonical_sha = sha256_file(canonical_path)
        if row.get('size_bytes') != canonical_path.stat().st_size or row.get('sha256') != canonical_sha:
            external_handoff_issues.append(f'canonical manifest mismatch: {rel}')
        elif local_sha != canonical_sha:
            external_handoff_issues.append(f'active/canonical SHA mismatch: {rel}')
if external_handoff_issues:
    print('External cache handoff is NOT complete:')
    for issue in external_handoff_issues:
        print(' -', issue)
    if REQUIRE_EXTERNAL_CACHE_HANDOFF:
        raise RuntimeError(
            'External artifacts are not durably reusable yet. Keep the A100 runtime attached, '
            'resolve/retry the failed export or mirror publish, and rerun this cell.'
        )
else:
    print('External cache handoff: VERIFIED 15/15 active + canonical artifacts with manifest SHA256.')
    print('SAFE TO DISCONNECT A100: reconnect a CPU runtime, rerun Setup, then run External Protocol Gate.')

print('Experimental outputs are written under reports/revision/experimental.')

# Forensic run-history wrapper. The legacy helper writes live output while this
# wrapper gives every invocation a unique, durable stage/run_id log and retains
# the requested stable path as the latest-run convenience copy.
FORENSIC_RUN_HISTORY_CAPABILITY = 'stage_run_id_v1'
_forensic_base_run = run

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    import os as _forensic_os
    import shutil as _forensic_shutil
    import subprocess as _forensic_subprocess
    import time as _forensic_time
    import uuid as _forensic_uuid
    from datetime import datetime as _forensic_datetime, timezone as _forensic_timezone
    from pathlib import Path as _ForensicPath

    run_cwd = _ForensicPath(cwd) if cwd else _ForensicPath.cwd()
    if log_path is None:
        stable_log = run_cwd / 'reports' / 'revision' / 'logs' / 'notebook_command_latest.log'
    else:
        stable_log = _ForensicPath(log_path)
        if not stable_log.is_absolute():
            stable_log = run_cwd / stable_log
    stable_log.parent.mkdir(parents=True, exist_ok=True)
    stage = stable_log.stem
    run_id = _forensic_datetime.now(_forensic_timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ') + '-' + _forensic_uuid.uuid4().hex[:10]
    history_log = stable_log.parent / 'history' / stage / f'{run_id}.log'
    history_log.parent.mkdir(parents=True, exist_ok=True)

    canonical_root = globals().get('MIRROR_REVISION_ROOT')
    if canonical_root is None and 'DRIVE_ROOT' in globals():
        canonical_root = _ForensicPath(DRIVE_ROOT) / 'revision_artifacts' / 'reports' / 'revision'
    canonical_history = None
    if canonical_root is not None:
        canonical_root = _ForensicPath(canonical_root)
        canonical_history = canonical_root / 'logs' / 'history' / stage / f'{run_id}.log'
        canonical_history.parent.mkdir(parents=True, exist_ok=True)

    started = _forensic_datetime.now(_forensic_timezone.utc).isoformat()
    header = f'run_id={run_id}\nstage={stage}\nstarted_utc={started}\ncommand={cmd}\n--- output ---\n'
    history_log.write_text(header, encoding='utf-8')
    if canonical_history is not None:
        canonical_history.write_text(header, encoding='utf-8')

    return_code = -1
    caught = None
    completed = None
    process = None
    try:
        process = _forensic_subprocess.Popen(
            cmd,
            shell=isinstance(cmd, str),
            cwd=str(run_cwd),
            stdout=_forensic_subprocess.PIPE,
            stderr=_forensic_subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        with history_log.open('a', encoding='utf-8') as local_handle:
            canonical_handle = (
                canonical_history.open('a', encoding='utf-8')
                if canonical_history is not None
                else None
            )
            try:
                for line in process.stdout or ():
                    print(line, end='', flush=True)
                    local_handle.write(line)
                    local_handle.flush()
                    if canonical_handle is not None:
                        canonical_handle.write(line)
                        canonical_handle.flush()
                return_code = int(process.wait())
                local_handle.flush()
                _forensic_os.fsync(local_handle.fileno())
                if canonical_handle is not None:
                    canonical_handle.flush()
                    _forensic_os.fsync(canonical_handle.fileno())
            finally:
                if canonical_handle is not None:
                    canonical_handle.close()
        completed = _forensic_subprocess.CompletedProcess(cmd, return_code)
    except BaseException as exc:
        caught = exc
        return_code = int(getattr(exc, 'returncode', -1))
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except Exception:
                process.kill()
                process.wait()
    finally:
        footer = (
            '\n--- end ---\n'
            f'ended_utc={_forensic_datetime.now(_forensic_timezone.utc).isoformat()}\n'
            f'return_code={return_code}\n'
        )
        with history_log.open('a', encoding='utf-8') as handle:
            handle.write(footer)
            handle.flush()
            _forensic_os.fsync(handle.fileno())
        if canonical_history is not None:
            # The underlying helper streams to this same canonical history path
            # when supported; append the footer or refresh from the local copy.
            try:
                _forensic_shutil.copy2(history_log, canonical_history)
            except Exception as exc:
                print('WARNING: durable history refresh failed:', exc)
        try:
            _forensic_shutil.copy2(history_log, stable_log)
            if canonical_root is not None:
                try:
                    revision_base = (_ForensicPath(globals().get('REPO_DIR', run_cwd)) / 'reports' / 'revision').resolve()
                    relative = stable_log.resolve().relative_to(revision_base)
                except (ValueError, TypeError):
                    relative = _ForensicPath('logs') / stable_log.name
                canonical_stable = canonical_root / relative
                canonical_stable.parent.mkdir(parents=True, exist_ok=True)
                _forensic_shutil.copy2(history_log, canonical_stable)
        except Exception as exc:
            print('WARNING: latest log refresh failed:', exc)
    print('Run history log:', history_log)
    if canonical_history is not None:
        print('Durable run history log:', canonical_history)
    if caught is not None:
        raise caught
    if check and return_code:
        raise _forensic_subprocess.CalledProcessError(return_code, cmd)
    return completed


## External Protocol Gate

Run this after external prediction artifacts exist. The expanded reviewer package gates PTB-XL, Georgia, and CPSC2021 together as dataset-specific mapped-task evaluations. Passing this gate permits conservative external mapped-task reporting only; it does not support unqualified zero-shot/external superiority.


In [ ]:
# Expanded external-evidence run: gate PTB-XL, Georgia, and CPSC2021 together.
# Rerun the gate instead of reusing stale cache, because Georgia may have newly restored/exported artifacts.
RUN_EXTERNAL_PROTOCOL_GATE = True
EXTERNAL_GATE_DATASETS = 'ptbxl,georgia,cpsc2021'
EXTERNAL_GATE_N_BOOT = 1000
EXTERNAL_GATE_STRICT = True
EXTERNAL_GATE_REUSE_EXISTING = True

# The gate consumes local active-repo files. Restore its exact signed inputs from the
# canonical Drive mirror so this cell is safe after a Colab disconnect or direct rerun.
EXTERNAL_GATE_INPUT_PATHS = [
    path
    for dataset in ['ptbxl', 'georgia', 'cpsc2021']
    for path in [
        Path(f'reports/revision/experimental/external/{dataset}/{dataset}_full_predictions.npz'),
        Path(f'reports/revision/experimental/external/{dataset}/{dataset}_full_slice_predictions.npz'),
        Path(f'reports/revision/experimental/external/{dataset}/{dataset}_full_prediction_summary.json'),
        Path(f'reports/revision/experimental/external/{dataset}/{dataset}_full_class_summary.csv'),
        Path(f'reports/revision/experimental/external/{dataset}/{dataset}_full_prediction_run_manifest.json'),
    ]
]
EXTERNAL_GATE_INPUT_PATHS.extend([
    Path('reports/revision/tables/table_georgia_snomed_code_inventory.csv'),
    Path('reports/revision/tables/table_cpsc2021_annotation_audit.csv'),
])
EXTERNAL_GATE_INPUT_PATHS.extend([
    Path('reports/revision/predictions/oof_final_ema_predictions.npz'),
    Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json'),
    Path('reports/revision/manifests/oof_final_ema_prediction_run_manifest.json'),
])
gate_missing_before = [path for path in EXTERNAL_GATE_INPUT_PATHS if not path.exists() or path.stat().st_size == 0]
gate_restore_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
gate_active_root = Path('reports/revision')
gate_restore_args = ' '.join(
    f'--include-path "{path.relative_to(gate_active_root).as_posix()}"'
    for path in EXTERNAL_GATE_INPUT_PATHS
)
print(f'Gate input authenticated restore: active_missing_before={len(gate_missing_before)} total={len(EXTERNAL_GATE_INPUT_PATHS)}.')
run(
    f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{gate_restore_root}" '
    f'--replace-mismatched {gate_restore_args}',
    log_path='reports/revision/logs/external_gate_input_restore.log',
)
gate_missing_after = [path for path in EXTERNAL_GATE_INPUT_PATHS if not path.exists() or path.stat().st_size == 0]
if gate_missing_after:
    raise FileNotFoundError(
        'External protocol gate inputs are absent from both the active repo and the authenticated canonical mirror: '
        + '; '.join(str(path) for path in gate_missing_after)
        + '. On an A100 High-RAM runtime, run the preceding Experimental External Prediction Commands cell '
        'with RUN_*_EXPORT="auto", wait for each immediate mirror publish, then rerun this gate.'
    )
print(f'External gate input contract: all {len(EXTERNAL_GATE_INPUT_PATHS)} artifacts are present in the active repo.')

# Fail before the long bootstrap when restored predictions were produced by a
# stale exporter/protocol/archive. This validator is CPU-only and has no Mamba/WFDB dependency.
from scripts.revision.external_reuse_contract import validate_external_prediction_reuse

gate_archive_names = {
    'ptbxl': ['PTB-XL.zip', 'ptb-xl.zip', 'ptbxl.zip'],
    'georgia': ['Georgia.zip'],
    'cpsc2021': ['cpsc2021.zip', 'CPSC2021.zip'],
}
gate_source_preflight = {}
for dataset in ['ptbxl', 'georgia', 'cpsc2021']:
    archive = next(
        (DRIVE_ROOT / name for name in gate_archive_names[dataset] if (DRIVE_ROOT / name).is_file()),
        None,
    )
    result = validate_external_prediction_reuse(
        dataset,
        revision_root=Path('reports/revision'),
        archive_path=archive,
        exporter_path=Path('scripts/revision/03_generate_external_predictions.py'),
        oof_path=Path('reports/revision/predictions/oof_final_ema_predictions.npz'),
        freeze_path=Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json'),
        archive_hash_cache_dir=gate_restore_root / 'manifests' / 'external_archive_hash_cache',
        threshold=0.5,
        q=3.0,
    )
    gate_source_preflight[dataset] = result
    print(
        f'External gate source-bound preflight: dataset={dataset} '
        f'ready={result.get("ready", False)} reasons={result.get("reasons", [])}'
    )
stale_gate_inputs = {
    dataset: result.get('reasons', [])
    for dataset, result in gate_source_preflight.items()
    if not result.get('ready', False)
}
if stale_gate_inputs:
    details = ' ; '.join(
        f'{dataset}=' + ','.join(reasons)
        for dataset, reasons in stale_gate_inputs.items()
    )
    raise RuntimeError(
        'External protocol gate stopped before bootstrap because restored prediction artifacts are stale: '
        + details
        + '. Reconnect an A100 High-RAM runtime and run Experimental External Prediction Commands '
        'with RUN_PTBXL_EXPORT=RUN_GEORGIA_EXPORT=RUN_CPSC2021_EXPORT="auto". '
        'Wait for External cache handoff: VERIFIED before returning to this CPU gate cell.'
    )

external_gate_command = (
    'python -u scripts/revision/18_external_protocol_gate.py '
    f'--dataset all --expected-checkpoint-kind final_ema '
    f'--threshold 0.5 --n-bins 15 --n-boot {EXTERNAL_GATE_N_BOOT}'
)
if EXTERNAL_GATE_STRICT:
    external_gate_command += ' --strict'
if EXTERNAL_GATE_REUSE_EXISTING:
    external_gate_command += ' --reuse-existing'
if EXTERNAL_GATE_DATASETS.strip() and EXTERNAL_GATE_DATASETS.strip().lower() != 'all':
    parts = [item.strip() for item in EXTERNAL_GATE_DATASETS.split(',') if item.strip()]
    external_gate_command = (
        'python -u scripts/revision/18_external_protocol_gate.py '
        + ' '.join(f'--dataset {item}' for item in parts)
        + f' --expected-checkpoint-kind final_ema --threshold 0.5 --n-bins 15 --n-boot {EXTERNAL_GATE_N_BOOT}'
    )
    if EXTERNAL_GATE_STRICT:
        external_gate_command += ' --strict'
    if EXTERNAL_GATE_REUSE_EXISTING:
        external_gate_command += ' --reuse-existing'

GATE_OUTPUT_PATHS = [
    Path('reports/revision/metrics/external_protocol_gate_summary.csv'),
    *[
        path
        for dataset in ['ptbxl', 'georgia', 'cpsc2021']
        for path in [
            Path(f'reports/revision/metrics/external_{dataset}_protocol_gate.json'),
            Path(f'reports/revision/tables/table_external_{dataset}_label_mapping.csv'),
            Path(f'reports/revision/tables/table_external_{dataset}_metrics.csv'),
            Path(f'reports/revision/manifests/external_{dataset}_protocol_gate_manifest.json'),
        ]
    ],
]
gate_publish_args = ' '.join(
    f'--include-path "{path.relative_to(Path("reports/revision")).as_posix()}"'
    for path in GATE_OUTPUT_PATHS
)

if RUN_EXTERNAL_PROTOCOL_GATE:
    run(
        external_gate_command,
        log_path='reports/revision/logs/external_protocol_gate.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --source-conflict-policy source {gate_publish_args} --mirror-root "{gate_restore_root}"',
        log_path='reports/revision/logs/external_protocol_gate_immediate_mirror_publish.log',
    )
else:
    print('External protocol gate disabled. For the expanded run, set RUN_EXTERNAL_PROTOCOL_GATE=True with EXTERNAL_GATE_DATASETS=\'ptbxl,georgia,cpsc2021\'.')
    print('Georgia/CPSC2021 will be claimable only if their dataset-specific gates pass.')
    print('Planned command:', external_gate_command)

for dataset in ['ptbxl', 'georgia', 'cpsc2021']:
    for path in [
        Path(f'reports/revision/metrics/external_{dataset}_protocol_gate.json'),
        Path(f'reports/revision/tables/table_external_{dataset}_label_mapping.csv'),
        Path(f'reports/revision/tables/table_external_{dataset}_metrics.csv'),
        Path(f'reports/revision/manifests/external_{dataset}_protocol_gate_manifest.json'),
    ]:
        print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
summary = Path('reports/revision/metrics/external_protocol_gate_summary.csv')
print(f'{summary}: exists={summary.exists()} size={summary.stat().st_size if summary.exists() else None}')
if summary.exists():
    import pandas as pd
    gate_summary = pd.read_csv(summary)
    display(gate_summary[['dataset', 'status', 'protocol_gate_passed', 'manuscript_ready', 'issue_count', 'reused_existing', 'gate_cache_key']])


## Deferred Georgia/CPSC2021 Protocol Gates

Legacy blocker-only gate. Keep disabled for the expanded run because the main External Protocol Gate now covers PTB-XL, Georgia, and CPSC2021. Enable only when intentionally recording Georgia/CPSC2021 as deferred without running or reusing export artifacts.


In [ ]:
DEFERRED_EXTERNAL_GATE_DATASETS = 'georgia,cpsc2021'
RUN_DEFERRED_EXTERNAL_GATES = False
DEFERRED_EXTERNAL_GATE_STRICT = False
DEFERRED_EXTERNAL_GATE_REUSE_EXISTING = True
DEFERRED_EXTERNAL_GATE_N_BOOT = 1000

_deferred_parts = [item.strip() for item in DEFERRED_EXTERNAL_GATE_DATASETS.split(',') if item.strip()]
deferred_gate_command = (
    'python -u scripts/revision/18_external_protocol_gate.py '
    + ' '.join(f'--dataset {item}' for item in _deferred_parts)
    + f' --expected-checkpoint-kind final_ema --threshold 0.5 --n-bins 15 --n-boot {DEFERRED_EXTERNAL_GATE_N_BOOT}'
    + ' --out-summary reports/revision/metrics/external_deferred_protocol_gate_summary.csv'
)
if DEFERRED_EXTERNAL_GATE_STRICT:
    deferred_gate_command += ' --strict'
if DEFERRED_EXTERNAL_GATE_REUSE_EXISTING:
    deferred_gate_command += ' --reuse-existing'

print('Deferred external gate command:', deferred_gate_command)
print('Deferred gate is disabled because the main External Protocol Gate cell now covers Georgia/CPSC2021. Enable only for a separate blocker-only summary.')
if RUN_DEFERRED_EXTERNAL_GATES:
    run(deferred_gate_command, log_path='reports/revision/logs/external_deferred_protocol_gate.log')
else:
    print('Deferred Georgia/CPSC2021 gate disabled. Enable only to record blocker/deferred artifacts without forcing exports.')

for dataset in _deferred_parts:
    for path in [
        Path(f'reports/revision/metrics/external_{dataset}_protocol_gate.json'),
        Path(f'reports/revision/tables/table_external_{dataset}_label_mapping.csv'),
        Path(f'reports/revision/tables/table_external_{dataset}_metrics.csv'),
        Path(f'reports/revision/manifests/external_{dataset}_protocol_gate_manifest.json'),
    ]:
        print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
summary = Path('reports/revision/metrics/external_deferred_protocol_gate_summary.csv')
print(f'{summary}: exists={summary.exists()} size={summary.stat().st_size if summary.exists() else None}')
if summary.exists():
    display(pd.read_csv(summary))


## Few-Shot External Adaptation Gate

Optional leakage-audited score-calibration sensitivity analysis. Run only after a dataset-specific external protocol gate has passed. It leaves model weights unchanged and must not be described as general few-shot transfer.


In [ ]:
# Legacy v1 row-split score calibration is retained only for provenance.
# It is not group-safe and must not be used as a few-shot claim.
RUN_LEGACY_ROW_SPLIT_SCORE_CALIBRATION = False
if RUN_LEGACY_ROW_SPLIT_SCORE_CALIBRATION:
    raise RuntimeError(
        'Legacy row-split score calibration is intentionally disabled. '
        'Use the group-safe and true-head-adaptation cells below.'
    )
print('Legacy row-split score-calibration cell disabled. It remains non-claim-ready provenance only.')


## PTB-XL Fold 9 Adaptation-Pool Export

A100-only when the five source artifacts are not already authenticated in the canonical mirror. Fold 9 is the independent target-label adaptation pool; fold 10 remains untouched test data.


In [ ]:
# PTB-XL follows the official convention: fold 9 is target adaptation data and fold 10 is held-out test data.
# This is GPU inference only. Reuse returns immediately when the verified artifact exists.
RUN_PTBXL_FOLD9_EXPORT = 'auto'
PTBXL_FOLD9_BATCH_SIZE = 128
PTBXL_FOLD9_REQUIRED = [
    Path('reports/revision/experimental/external/ptbxl/ptbxl_full_fold9_predictions.npz'),
    Path('reports/revision/experimental/external/ptbxl/ptbxl_full_fold9_slice_predictions.npz'),
    Path('reports/revision/experimental/external/ptbxl/ptbxl_full_fold9_prediction_summary.json'),
    Path('reports/revision/experimental/external/ptbxl/ptbxl_full_fold9_prediction_run_manifest.json'),
    Path('reports/revision/experimental/external/ptbxl/ptbxl_full_fold9_class_summary.csv'),
]
if '_restore_report_artifact' in globals():
    ptbxl_fold9_restore_roots = globals().get('restore_source_roots', [
        DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision',
    ])
    ptbxl_fold9_restore = [
        _restore_report_artifact(path, ptbxl_fold9_restore_roots)
        for path in PTBXL_FOLD9_REQUIRED
    ]
    print('PTB-XL fold 9 targeted restore:', ptbxl_fold9_restore)
else:
    print('PTB-XL fold 9 targeted restore helper is unavailable; run Experimental External Prediction Commands first.')
def _ptbxl_fold9_contract_ready():
    if not all(path.exists() and path.stat().st_size > 0 for path in PTBXL_FOLD9_REQUIRED):
        return False
    try:
        from scripts.revision.common import sha256_file
        manifest_path = PTBXL_FOLD9_REQUIRED[3]
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        canonical = {
            'oof_sha256': sha256_file(Path('reports/revision/predictions/oof_final_ema_predictions.npz')),
            'freeze_sha256': sha256_file(Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json')),
        }
        runner = Path('scripts/revision/03_generate_external_predictions.py')
        if manifest.get('canonical_contract') != canonical:
            return False
        if manifest.get('runner_sha256') != sha256_file(runner):
            return False
        if manifest.get('dataset') != 'ptbxl' or manifest.get('split_ids') != ['ptbxl_fold9']:
            return False
        if '_strat_folds_9' not in str(manifest.get('protocol', '')):
            return False
        output_rows = manifest.get('outputs') or {}
        for path in (PTBXL_FOLD9_REQUIRED[0], PTBXL_FOLD9_REQUIRED[1], PTBXL_FOLD9_REQUIRED[2], PTBXL_FOLD9_REQUIRED[4]):
            row = output_rows.get(path.name) or {}
            if row.get('sha256') != sha256_file(path):
                return False
        with np.load(PTBXL_FOLD9_REQUIRED[0], allow_pickle=False) as payload:
            if set(np.asarray(payload['split_id']).astype(str)) != {'ptbxl_fold9'}:
                return False
        return True
    except Exception as exc:
        print('PTB-XL fold 9 reuse contract rejected:', repr(exc))
        return False

ptbxl_fold9_ready = _ptbxl_fold9_contract_ready()
ptbxl_fold9_should_run = RUN_PTBXL_FOLD9_EXPORT is True or (
    str(RUN_PTBXL_FOLD9_EXPORT).lower() == 'auto' and not ptbxl_fold9_ready
)
print('PTB-XL fold 9 adaptation-pool ready:', ptbxl_fold9_ready)
if ptbxl_fold9_should_run:
    try:
        require_gpu_inference_runtime('PTB-XL fold 9 Full-model export')
    except RuntimeError as exc:
        if RUN_PTBXL_FOLD9_EXPORT is True:
            raise
        ptbxl_fold9_should_run = False
        print('DEFERRED PTB-XL fold 9 export:', exc)
        print('Use A100 High-RAM to create this adaptation-pool artifact. The completed external protocol gates remain valid.')
if ptbxl_fold9_should_run:
    run(
        'python -u scripts/revision/03_generate_external_predictions.py '
        '--dataset ptbxl --ptbxl-folds 9 --output-tag fold9 '
        f'--checkpoint-kind final_ema --batch-size {PTBXL_FOLD9_BATCH_SIZE} --allow-experimental',
        log_path='reports/revision/logs/ptbxl_fold9_generate_predictions.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{DRIVE_ROOT / "revision_artifacts" / "reports" / "revision"}"',
        log_path='reports/revision/logs/ptbxl_fold9_immediate_mirror_publish.log',
    )
else:
    print('Reusing PTB-XL fold 9 external Full-model artifacts.' if ptbxl_fold9_ready else 'PTB-XL fold 9 export deferred; no reusable adaptation-pool artifact exists yet.')
for path in PTBXL_FOLD9_REQUIRED:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## PTB-XL Adaptation Analysis Lock

CPU-only reproducibility lock for official fold 9 adaptation and fold 10 evaluation. This post-initial-review lock prevents silent changes during reruns; it is not a preregistration.


In [ ]:
# PTBXL_ADAPTATION_ANALYSIS_LOCK_CELL_V1
PTBXL_ADAPTATION_LOCK = Path('reports/revision/manifests/ptbxl_adaptation_analysis_lock.json')
if '_restore_report_artifact' in globals():
    lock_restore_roots = globals().get('restore_source_roots', [MIRROR_REVISION_ROOT])
    print('PTB-XL analysis-lock restore:', _restore_report_artifact(PTBXL_ADAPTATION_LOCK, lock_restore_roots))
run(
    'python -u scripts/revision/51_ptbxl_adaptation_analysis_lock.py '
    '--models full,resnet,raw_mamba,transformer --fractions 0,0.01,0.05,0.10 '
    '--primary-fraction 0.10 --seeds 42,43,44,45,46 --threshold 0.5 '
    '--n-bins 15 --n-boot 1000 --head-c 1.0 --max-iter 5000',
    log_path='reports/revision/logs/ptbxl_adaptation_analysis_lock.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing full '
    f'--source-conflict-policy source --include-path "manifests/ptbxl_adaptation_analysis_lock.json" '
    f'--mirror-root "{MIRROR_REVISION_ROOT}"',
    log_path='reports/revision/logs/ptbxl_adaptation_analysis_lock_mirror_publish.log',
)
ptbxl_analysis_lock = json.loads(PTBXL_ADAPTATION_LOCK.read_text(encoding='utf-8'))
print('PTB-XL analysis lock:', ptbxl_analysis_lock['protocol_sha256'])
print('Temporal qualification:', ptbxl_analysis_lock['temporal_qualification'])


## External Learned-Comparator Zero-Target-Label Inference

A100-only when fold caches are missing. ResNet1D/CNN, Raw Mamba, and a completed Transformer ECG baseline are applied without target-label training. PTB-XL, Georgia, and CPSC2021 are retained as separate tasks. Notebook 04 owns the required in-domain summaries, paired contracts, and five checkpoints per model.


In [ ]:
# GPU required. The script caches each dataset/comparator/fold directly on Drive, so a disconnect resumes.
RUN_EXTERNAL_LEARNED_COMPARATORS = 'auto'
RUN_PTBXL_FOLD9_COMPARATORS = 'auto'
EXTERNAL_COMPARATOR_DATASETS = 'ptbxl,georgia,cpsc2021'
EXTERNAL_COMPARATOR_MODELS = 'resnet,raw_mamba'
REQUIRE_TRANSFORMER_EXTERNAL_EVIDENCE = True
EXTERNAL_COMPARATOR_BATCH_SIZE = 256
EXTERNAL_COMPARATOR_NUM_WORKERS = 0
EXTERNAL_COMPARATOR_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'predictions' / 'external_comparator_folds'
EXTERNAL_COMPARATOR_CHECKPOINT_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'experimental'

def _checkpoint_dir(name):
    candidates = [
        EXTERNAL_COMPARATOR_CHECKPOINT_ROOT / name,
        Path('reports/revision/experimental') / name,
    ]
    return next((path for path in candidates if path.exists()), candidates[0])

resnet_checkpoint_dir = _checkpoint_dir('resnet1d_cnn_checkpoints')
raw_mamba_checkpoint_dir = _checkpoint_dir('raw_mamba_checkpoints')
transformer_checkpoint_dir = _checkpoint_dir('transformer_ecg_checkpoints')

# The external runner validates in-domain OOF/paired contracts. Restore only those small JSON files
# when this notebook resumed in a fresh local runtime; checkpoints are read directly from Drive.
comparator_contract_relpaths = [
    Path('metrics/resnet1d_cnn_baseline_summary.json'),
    Path('manifests/resnet1d_cnn_baseline_manifest.json'),
    Path('metrics/paired_full_vs_resnet_comparison.json'),
    Path('metrics/raw_mamba_baseline_summary.json'),
    Path('manifests/raw_mamba_baseline_manifest.json'),
    Path('metrics/paired_full_vs_raw_mamba_comparison.json'),
    Path('metrics/transformer_ecg_baseline_summary.json'),
    Path('manifests/transformer_ecg_baseline_manifest.json'),
    Path('metrics/paired_full_vs_transformer_comparison.json'),
    Path('predictions/resnet1d_cnn_oof_predictions.npz'),
    Path('tables/table_paired_full_vs_resnet.csv'),
    Path('metrics/paired_full_vs_resnet_bootstrap_samples.csv'),
    Path('manifests/paired_full_vs_resnet_manifest.json'),
    Path('predictions/raw_mamba_oof_predictions.npz'),
    Path('tables/table_paired_full_vs_raw_mamba.csv'),
    Path('metrics/paired_full_vs_raw_mamba_bootstrap_samples.csv'),
    Path('manifests/paired_full_vs_raw_mamba_manifest.json'),
    Path('predictions/transformer_ecg_oof_predictions.npz'),
    Path('tables/table_paired_full_vs_transformer.csv'),
    Path('metrics/paired_full_vs_transformer_bootstrap_samples.csv'),
    Path('manifests/paired_full_vs_transformer_manifest.json'),
]
if '_restore_report_artifact' in globals():
    comparator_restore_roots = globals().get('restore_source_roots', [
        DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision',
    ])
    comparator_restore_rows = [
        _restore_report_artifact(Path('reports/revision') / rel, comparator_restore_roots)
        for rel in comparator_contract_relpaths
    ]
    print('In-domain comparator contract restore:', comparator_restore_rows)


# CPU-only: a metadata-only freeze refresh changes the exact freeze SHA but does
# not require baseline retraining. Rebuild only stale paired tables/JSONs from
# the existing same-record OOF prediction NPZs, then bind them to the strict
# current freeze contract before any external GPU inference.
run(
    'python -u scripts/revision/50_refresh_in_domain_paired_contracts.py '
    '--models resnet,raw_mamba,transformer --n-boot 1000 --allow-incomplete',
    log_path='reports/revision/logs/in_domain_paired_contract_refresh.log',
)
PAIRED_REFRESH_OUTPUTS = [
    Path('reports/revision/metrics/in_domain_paired_contract_refresh.json'),
    Path('reports/revision/metrics/paired_full_vs_resnet_comparison.json'),
    Path('reports/revision/tables/table_paired_full_vs_resnet.csv'),
    Path('reports/revision/metrics/paired_full_vs_resnet_bootstrap_samples.csv'),
    Path('reports/revision/manifests/paired_full_vs_resnet_manifest.json'),
    Path('reports/revision/metrics/paired_full_vs_raw_mamba_comparison.json'),
    Path('reports/revision/tables/table_paired_full_vs_raw_mamba.csv'),
    Path('reports/revision/metrics/paired_full_vs_raw_mamba_bootstrap_samples.csv'),
    Path('reports/revision/manifests/paired_full_vs_raw_mamba_manifest.json'),
    Path('reports/revision/metrics/paired_full_vs_transformer_comparison.json'),
    Path('reports/revision/tables/table_paired_full_vs_transformer.csv'),
    Path('reports/revision/metrics/paired_full_vs_transformer_bootstrap_samples.csv'),
    Path('reports/revision/manifests/paired_full_vs_transformer_manifest.json'),
]
paired_refresh_publish_args = ' '.join(
    f'--include-path "{path.relative_to(Path("reports/revision")).as_posix()}"'
    for path in PAIRED_REFRESH_OUTPUTS if path.exists() and path.stat().st_size > 0
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing full '
    f'--source-conflict-policy source {paired_refresh_publish_args} --mirror-root "{MIRROR_REVISION_ROOT}"',
    log_path='reports/revision/logs/in_domain_paired_contract_refresh_mirror_publish.log',
)
paired_refresh_status = json.loads(
    Path('reports/revision/metrics/in_domain_paired_contract_refresh.json').read_text(encoding='utf-8')
)
paired_refresh_complete_models = {
    row['model'] for row in paired_refresh_status.get('models', [])
    if row.get('status') == 'complete'
}
print('Exact current paired models:', sorted(paired_refresh_complete_models))

BASELINE_CHECKPOINT_PROTOCOLS_02 = {
    'resnet': 'resnet1d_cnn_raw_same_folds_power_mean_v2_q3_threshold_0.5',
    'raw_mamba': 'raw_mamba_retrained_weighted_bce_same_folds_power_mean_v2_q3_threshold_0.5',
    'transformer': 'transformer_ecg_raw_same_folds_power_mean_v2_q3_threshold_0.5',
}
BASELINE_CHECKPOINT_MANIFESTS_02 = {
    'resnet': Path('reports/revision/manifests/resnet1d_cnn_baseline_manifest.json'),
    'raw_mamba': Path('reports/revision/manifests/raw_mamba_baseline_manifest.json'),
    'transformer': Path('reports/revision/manifests/transformer_ecg_baseline_manifest.json'),
}
BASELINE_CHECKPOINT_NAMES_02 = {
    'resnet': 'fold{fold}_resnet1d_cnn_final.pt',
    'raw_mamba': 'fold{fold}_raw_mamba_final_ema.pt',
    'transformer': 'fold{fold}_transformer_ecg_final.pt',
}

def _baseline_checkpoint_contract_status_02(model, checkpoint_dir):
    manifest_path = BASELINE_CHECKPOINT_MANIFESTS_02[model]
    issues = []
    if not manifest_path.exists() or manifest_path.stat().st_size == 0:
        return False, [f'missing manifest: {manifest_path}']
    try:
        payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    except Exception as exc:
        return False, [f'unreadable manifest: {manifest_path}: {exc}']
    if payload.get('protocol') != BASELINE_CHECKPOINT_PROTOCOLS_02[model]:
        issues.append(f'protocol mismatch: {payload.get("protocol")!r}')
    contract = payload.get('checkpoint_contract') or {}
    rows = contract.get('checkpoints') or []
    by_fold = {int(row['fold']): row for row in rows if isinstance(row, dict) and row.get('fold') is not None}
    if contract.get('status') != 'complete':
        issues.append(f'checkpoint_contract status={contract.get("status")!r}')
    if sorted(by_fold) != [1, 2, 3, 4, 5]:
        issues.append(f'declared checkpoint folds={sorted(by_fold)}')
    for fold in range(1, 6):
        expected_path = (Path(checkpoint_dir) / BASELINE_CHECKPOINT_NAMES_02[model].format(fold=fold)).resolve()
        row = by_fold.get(fold)
        if not expected_path.exists() or expected_path.stat().st_size == 0:
            issues.append(f'fold {fold} checkpoint missing: {expected_path}')
            continue
        if row is None:
            continue
        declared_path = Path(str(row.get('path') or '')).expanduser()
        if not declared_path.is_absolute():
            declared_path = REPO_DIR / declared_path
        if declared_path.resolve() != expected_path:
            issues.append(f'fold {fold} manifest path mismatch: {declared_path.resolve()} != {expected_path}')
        if int(row.get('size_bytes', -1)) != expected_path.stat().st_size:
            issues.append(f'fold {fold} manifest size mismatch')
        if len(str(row.get('sha256') or '')) != 64:
            issues.append(f'fold {fold} manifest SHA missing')
    return not issues, issues

resnet_checkpoint_contract_ready, resnet_checkpoint_contract_issues = _baseline_checkpoint_contract_status_02('resnet', resnet_checkpoint_dir)
raw_mamba_checkpoint_contract_ready, raw_mamba_checkpoint_contract_issues = _baseline_checkpoint_contract_status_02('raw_mamba', raw_mamba_checkpoint_dir)
transformer_checkpoint_contract_ready, transformer_checkpoint_contract_issues = _baseline_checkpoint_contract_status_02('transformer', transformer_checkpoint_dir)
for label, ready, issues in [
    ('ResNet1D/CNN', resnet_checkpoint_contract_ready, resnet_checkpoint_contract_issues),
    ('Raw Mamba', raw_mamba_checkpoint_contract_ready, raw_mamba_checkpoint_contract_issues),
    ('Transformer ECG', transformer_checkpoint_contract_ready, transformer_checkpoint_contract_issues),
]:
    print(f'{label} checkpoint contract ready={ready}')
    for issue in issues:
        print('  -', issue)

comparator_models = [item.strip() for item in EXTERNAL_COMPARATOR_MODELS.split(',') if item.strip()]
transformer_prerequisites = [
    Path('reports/revision/metrics/transformer_ecg_baseline_summary.json'),
    Path('reports/revision/manifests/transformer_ecg_baseline_manifest.json'),
    Path('reports/revision/metrics/paired_full_vs_transformer_comparison.json'),
] + [
    transformer_checkpoint_dir / f'fold{fold}_transformer_ecg_final.pt' for fold in range(1, 6)
]
transformer_files_ready = all(path.exists() and path.stat().st_size > 0 for path in transformer_prerequisites)
if transformer_files_ready and transformer_checkpoint_contract_ready and 'transformer' in paired_refresh_complete_models:
    comparator_models.append('transformer')
    print('Transformer is in-domain complete; including it in external comparator inference.')
else:
    print('Transformer external inference deferred until its in-domain OOF/paired/checkpoint gate is complete.')
    if REQUIRE_TRANSFORMER_EXTERNAL_EVIDENCE:
        print('  Missing Transformer prerequisites:', [str(path) for path in transformer_prerequisites if not path.exists() or path.stat().st_size == 0])
        print('  Transformer checkpoint-contract issues:', transformer_checkpoint_contract_issues)

learned_in_domain_prerequisites = [
    Path('reports/revision/metrics/resnet1d_cnn_baseline_summary.json'),
    Path('reports/revision/manifests/resnet1d_cnn_baseline_manifest.json'),
    Path('reports/revision/metrics/paired_full_vs_resnet_comparison.json'),
    Path('reports/revision/metrics/raw_mamba_baseline_summary.json'),
    Path('reports/revision/manifests/raw_mamba_baseline_manifest.json'),
    Path('reports/revision/metrics/paired_full_vs_raw_mamba_comparison.json'),
] + [
    resnet_checkpoint_dir / f'fold{fold}_resnet1d_cnn_final.pt' for fold in range(1, 6)
] + [
    raw_mamba_checkpoint_dir / f'fold{fold}_raw_mamba_final_ema.pt' for fold in range(1, 6)
]
learned_in_domain_files_ready = all(path.exists() and path.stat().st_size > 0 for path in learned_in_domain_prerequisites)
learned_in_domain_ready = (learned_in_domain_files_ready and resnet_checkpoint_contract_ready and raw_mamba_checkpoint_contract_ready and {'resnet', 'raw_mamba'}.issubset(paired_refresh_complete_models))
missing_learned_prerequisites = [str(path) for path in learned_in_domain_prerequisites if not path.exists() or path.stat().st_size == 0]
missing_learned_prerequisites += [f'ResNet checkpoint contract: {issue}' for issue in resnet_checkpoint_contract_issues]
missing_learned_prerequisites += [f'Raw Mamba checkpoint contract: {issue}' for issue in raw_mamba_checkpoint_contract_issues]
print('In-domain ResNet/Raw-Mamba prerequisites ready:', learned_in_domain_ready)

datasets = [item.strip() for item in EXTERNAL_COMPARATOR_DATASETS.split(',') if item.strip()]
def _comparator_artifacts(dataset, model, tag=''):
    model_stem = {'resnet': 'resnet1d_cnn', 'raw_mamba': 'raw_mamba', 'transformer': 'transformer_ecg'}[model]
    suffix = f'_{tag}' if tag else ''
    stem = f'{dataset}_{model_stem}{suffix}'
    return [
        Path(f'reports/revision/experimental/external/{dataset}/{stem}_predictions.npz'),
        Path(f'reports/revision/experimental/external/{dataset}/{stem}_slice_predictions.npz'),
        Path(f'reports/revision/metrics/external_{stem}_summary.json'),
        Path(f'reports/revision/tables/table_external_{stem}_class_metrics.csv'),
        Path(f'reports/revision/manifests/external_{stem}_manifest.json'),
    ]

test_outputs = [path for dataset in datasets for model in comparator_models for path in _comparator_artifacts(dataset, model)]
fold9_outputs = [path for model in comparator_models for path in _comparator_artifacts('ptbxl', model, 'fold9')]
if '_restore_report_artifact' in globals():
    comparator_external_restore_roots = globals().get('restore_source_roots', [MIRROR_REVISION_ROOT])
    comparator_external_restore = [
        _restore_report_artifact(path, comparator_external_restore_roots)
        for path in test_outputs + fold9_outputs
    ]
    restored = sum(row.get('status') == 'restored' for row in comparator_external_restore)
    print(f'External comparator targeted restore: restored={restored} total={len(comparator_external_restore)}')
test_ready = all(path.exists() and path.stat().st_size > 0 for path in test_outputs)
fold9_ready = all(path.exists() and path.stat().st_size > 0 for path in fold9_outputs)
# Auto invokes the runner's lightweight final-artifact provenance verifier only after Notebook 04
# has completed the required in-domain ResNet/Raw-Mamba contracts. This avoids a false failure when
# Notebook 02 is run before Notebook 04 in the normal top-to-bottom sequence.
requested_test = RUN_EXTERNAL_LEARNED_COMPARATORS is True or str(RUN_EXTERNAL_LEARNED_COMPARATORS).lower() == 'auto'
requested_fold9 = RUN_PTBXL_FOLD9_COMPARATORS is True or str(RUN_PTBXL_FOLD9_COMPARATORS).lower() == 'auto'
if not learned_in_domain_ready:
    if RUN_EXTERNAL_LEARNED_COMPARATORS is True or RUN_PTBXL_FOLD9_COMPARATORS is True:
        raise RuntimeError('External learned-comparator inference was forced, but Notebook 04 prerequisites are incomplete: ' + '; '.join(missing_learned_prerequisites))
    test_should_run = False
    fold9_should_run = False
    print('Deferred external learned-comparator inference. Run the latest Notebook 04 ResNet/Raw-Mamba/Transformer cells in aggregate/reuse mode, publish the refreshed manifests, then rerun this cell. Existing compatible checkpoints are reused and are not retrained.')
    print('Missing or stale contracts:', missing_learned_prerequisites)
else:
    test_should_run = requested_test
    fold9_should_run = requested_fold9
pending_external_comparator_test = test_should_run and not test_ready
pending_external_comparator_fold9 = fold9_should_run and not fold9_ready
if pending_external_comparator_test or pending_external_comparator_fold9:
    pending_parts = []
    if pending_external_comparator_test:
        pending_parts.append('test:' + ','.join(datasets))
    if pending_external_comparator_fold9:
        pending_parts.append('ptbxl_fold9')
    try:
        require_gpu_inference_runtime('External learned-comparator inference for ' + ';'.join(pending_parts))
    except RuntimeError as exc:
        forced_pending = (pending_external_comparator_test and RUN_EXTERNAL_LEARNED_COMPARATORS is True) or (pending_external_comparator_fold9 and RUN_PTBXL_FOLD9_COMPARATORS is True)
        if forced_pending:
            raise
        if pending_external_comparator_test:
            test_should_run = False
        if pending_external_comparator_fold9:
            fold9_should_run = False
        print('DEFERRED external learned-comparator inference:', exc)
        print('Reconnect an A100 High-RAM runtime; completed per-fold Drive caches will be reused.')
base = (
    'python -u scripts/revision/31_generate_external_comparator_predictions.py '
    f'--comparators {",".join(comparator_models)} --batch-size {EXTERNAL_COMPARATOR_BATCH_SIZE} '
    f'--num-workers {EXTERNAL_COMPARATOR_NUM_WORKERS} --device auto --amp --allow-tf32 --reuse-existing '
    f'--strict --fold-cache-dir "{EXTERNAL_COMPARATOR_CACHE_DIR}" '
    f'--resnet-checkpoint-dir "{resnet_checkpoint_dir}" '
    f'--raw-mamba-checkpoint-dir "{raw_mamba_checkpoint_dir}" '
    f'--transformer-checkpoint-dir "{transformer_checkpoint_dir}" --allow-experimental'
)
if test_should_run:
    run(
        base + ' ' + ' '.join(f'--dataset {dataset}' for dataset in datasets),
        log_path='reports/revision/logs/external_learned_comparators_test.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{DRIVE_ROOT / "revision_artifacts" / "reports" / "revision"}"',
        log_path='reports/revision/logs/external_learned_comparators_test_mirror_publish.log',
    )
else:
    print('Reusing test-split external learned-comparator predictions.' if test_ready else 'Test-split external learned-comparator inference deferred; artifacts are not yet reusable.')
if fold9_should_run:
    run(
        base + ' --dataset ptbxl --ptbxl-folds 9 --output-tag fold9',
        log_path='reports/revision/logs/external_learned_comparators_ptbxl_fold9.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{DRIVE_ROOT / "revision_artifacts" / "reports" / "revision"}"',
        log_path='reports/revision/logs/external_learned_comparators_ptbxl_fold9_mirror_publish.log',
    )
else:
    print('Reusing PTB-XL fold 9 learned-comparator predictions.' if fold9_ready else 'PTB-XL fold 9 learned-comparator inference deferred; artifacts are not yet reusable.')
for path in test_outputs + fold9_outputs:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## PTB-XL Fold 9/10 Patient and Unsupported-Only Audit

CPU-only audit of patient disjointness, common records across all four models, and a label-defined sensitivity analysis excluding records with no supported positive superclass.


In [ ]:
# PTBXL_FOLD_PROTOCOL_AUDIT_CELL_V1
RUN_PTBXL_FOLD_PROTOCOL_AUDIT = 'auto'
PTBXL_FOLD_PROTOCOL_N_BOOT = 1000
PTBXL_FOLD_PROTOCOL_CACHE = MIRROR_REVISION_ROOT / 'metrics' / 'ptbxl_fold_protocol_metric_cache'
PTBXL_FOLD_PROTOCOL_REQUIRED = [
    Path('reports/revision/metrics/ptbxl_fold_protocol_audit.json'),
    Path('reports/revision/tables/table_ptbxl_fold_protocol_audit.csv'),
    Path('reports/revision/tables/table_ptbxl_unsupported_only_sensitivity.csv'),
    Path('reports/revision/manifests/ptbxl_fold_protocol_audit_manifest.json'),
]
if '_restore_report_artifact' in globals():
    protocol_restore_roots = globals().get('restore_source_roots', [MIRROR_REVISION_ROOT])
    print('PTB-XL fold-protocol audit restore:', [
        _restore_report_artifact(path, protocol_restore_roots)
        for path in PTBXL_FOLD_PROTOCOL_REQUIRED
    ])
ptbxl_protocol_inputs = [
    path for model in ['full', 'resnet', 'raw_mamba', 'transformer']
    for path in (
        [Path('reports/revision/experimental/external/ptbxl/ptbxl_full_predictions.npz'), Path('reports/revision/experimental/external/ptbxl/ptbxl_full_fold9_predictions.npz')]
        if model == 'full'
        else [_comparator_artifacts('ptbxl', model)[0], _comparator_artifacts('ptbxl', model, 'fold9')[0]]
    )
]
ptbxl_protocol_inputs_ready = all(path.exists() and path.stat().st_size > 0 for path in ptbxl_protocol_inputs)
ptbxl_protocol_should_run = RUN_PTBXL_FOLD_PROTOCOL_AUDIT is True or (
    str(RUN_PTBXL_FOLD_PROTOCOL_AUDIT).lower() == 'auto' and ptbxl_protocol_inputs_ready
)
if RUN_PTBXL_FOLD_PROTOCOL_AUDIT is True and not ptbxl_protocol_inputs_ready:
    raise FileNotFoundError('PTB-XL fold-protocol audit was forced but inputs are missing: ' + '; '.join(str(path) for path in ptbxl_protocol_inputs if not path.exists() or path.stat().st_size == 0))
if ptbxl_protocol_should_run:
    run(
        'python -u scripts/revision/52_ptbxl_fold_protocol_audit.py '
        '--models full,resnet,raw_mamba,transformer --threshold 0.5 --n-bins 15 '
        f'--n-boot {PTBXL_FOLD_PROTOCOL_N_BOOT} --strict --reuse-existing '
        f'--analysis-lock "{PTBXL_ADAPTATION_LOCK}" --metric-cache-dir "{PTBXL_FOLD_PROTOCOL_CACHE}"',
        log_path='reports/revision/logs/ptbxl_fold_protocol_audit.log',
    )
    audit_publish_args = ' '.join(
        f'--include-path "{path.relative_to(Path("reports/revision")).as_posix()}"'
        for path in PTBXL_FOLD_PROTOCOL_REQUIRED
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing full '
        f'--source-conflict-policy source {audit_publish_args} --mirror-root "{MIRROR_REVISION_ROOT}"',
        log_path='reports/revision/logs/ptbxl_fold_protocol_audit_mirror_publish.log',
    )
else:
    print('PTB-XL fold-protocol audit deferred until all Full/ResNet/Raw-Mamba/Transformer fold9 and fold10 predictions exist.')
for path in PTBXL_FOLD_PROTOCOL_REQUIRED:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## Paired External Comparator Audit

CPU-only paired group bootstrap. Patients/source records are resampled as intact clusters and no metric is pooled across datasets.


In [ ]:
# CPU-only. Bootstrap caches are persisted on Drive and keyed by input provenance.
RUN_EXTERNAL_PAIRED_COMPARATOR_AUDIT = 'auto'
EXTERNAL_PAIRED_DATASETS = 'ptbxl,georgia,cpsc2021'
EXTERNAL_PAIRED_N_BOOT = 1000
EXTERNAL_PAIRED_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'metrics' / 'external_comparator_metric_cache'
EXTERNAL_PAIRED_REQUIRED = [
    Path('reports/revision/metrics/external_comparator_paired_summary.json'),
    Path('reports/revision/tables/table_external_comparator_paired.csv'),
    Path('reports/revision/metrics/external_comparator_paired_bootstrap_samples.csv'),
    Path('reports/revision/manifests/external_comparator_paired_manifest.json'),
]
if '_restore_report_artifact' in globals():
    external_paired_restore_roots = globals().get('restore_source_roots', [
        DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision',
    ])
    print('External paired-result targeted restore:', [
        _restore_report_artifact(path, external_paired_restore_roots)
        for path in EXTERNAL_PAIRED_REQUIRED
    ])
external_paired_ready = all(path.exists() and path.stat().st_size > 0 for path in EXTERNAL_PAIRED_REQUIRED)
external_paired_should_run = RUN_EXTERNAL_PAIRED_COMPARATOR_AUDIT is True or (
    str(RUN_EXTERNAL_PAIRED_COMPARATOR_AUDIT).lower() == 'auto'
)
external_paired_datasets = [item.strip() for item in EXTERNAL_PAIRED_DATASETS.split(',') if item.strip()]
external_paired_models = list(globals().get('comparator_models', ['resnet', 'raw_mamba']))
transformer_external_ready = all(
    Path(f'reports/revision/experimental/external/{dataset}/{dataset}_transformer_ecg_predictions.npz').exists()
    for dataset in external_paired_datasets
)
if transformer_external_ready and 'transformer' not in external_paired_models:
    external_paired_models.append('transformer')
print('External paired comparators:', external_paired_models)
external_pair_prediction_inputs = [
    Path(f'reports/revision/experimental/external/{dataset}/{dataset}_full_predictions.npz')
    for dataset in external_paired_datasets
] + [
    Path(f'reports/revision/experimental/external/{dataset}/{dataset}_{stem}_predictions.npz')
    for dataset in external_paired_datasets
    for stem in [{'resnet': 'resnet1d_cnn', 'raw_mamba': 'raw_mamba', 'transformer': 'transformer_ecg'}[model] for model in external_paired_models]
]
external_pair_inputs_ready = all(path.exists() and path.stat().st_size > 0 for path in external_pair_prediction_inputs)
if not external_pair_inputs_ready:
    if RUN_EXTERNAL_PAIRED_COMPARATOR_AUDIT is True:
        raise FileNotFoundError('External paired audit was forced, but comparator prediction inputs are missing: ' + '; '.join(str(path) for path in external_pair_prediction_inputs if not path.exists() or path.stat().st_size == 0))
    external_paired_should_run = False
    print('Deferred paired external audit until external learned-comparator predictions are complete.')
if external_paired_should_run:
    run(
        'python -u scripts/revision/32_paired_external_comparators.py '
        + ' '.join(f'--dataset {dataset}' for dataset in external_paired_datasets)
        + f' --comparators {",".join(external_paired_models)} '
        f'--threshold 0.5 --n-bins 15 --n-boot {EXTERNAL_PAIRED_N_BOOT} --strict --reuse-existing '
        f'--metric-cache-dir "{EXTERNAL_PAIRED_CACHE_DIR}"',
        log_path='reports/revision/logs/external_paired_comparators.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{DRIVE_ROOT / "revision_artifacts" / "reports" / "revision"}"',
        log_path='reports/revision/logs/external_paired_comparators_mirror_publish.log',
    )
else:
    print('Reusing paired dataset-specific external learned-comparator audit.' if external_paired_ready else 'Paired external comparator audit deferred; outputs are missing or stale.')
for path in EXTERNAL_PAIRED_REQUIRED:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## Group-Safe Score Calibration

CPU-only PTB-XL sensitivity analysis using fold 9 adaptation groups and fixed fold 10 test groups. Model weights remain unchanged; this is not true few-shot model adaptation.


In [ ]:
# CPU-only. This is monotonic score calibration of frozen Full-model probabilities, not weight adaptation.
RUN_GROUP_SAFE_SCORE_CALIBRATION = 'auto'
GROUP_SAFE_CALIBRATION_N_BOOT = 1000
GROUP_SAFE_CALIBRATION_PRIMARY_FRACTION = 0.10
GROUP_SAFE_CALIBRATION_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'metrics' / 'group_safe_score_calibration_metric_cache'
GROUP_SAFE_CALIBRATION_REQUIRED = [
    Path('reports/revision/metrics/group_safe_score_calibration_ptbxl_summary.csv'),
    Path('reports/revision/tables/table_group_safe_score_calibration_ptbxl.csv'),
    Path('reports/revision/metrics/group_safe_score_calibration_ptbxl_bootstrap.json'),
    Path('reports/revision/manifests/group_safe_score_calibration_ptbxl_splits.npz'),
    Path('reports/revision/tables/table_group_safe_score_calibration_ptbxl_coefficients.csv'),
    Path('reports/revision/manifests/group_safe_score_calibration_ptbxl_manifest.json'),
]
if '_restore_report_artifact' in globals():
    group_safe_restore_roots = globals().get('restore_source_roots', [
        DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision',
    ])
    print('Group-safe calibration targeted restore:', [
        _restore_report_artifact(path, group_safe_restore_roots)
        for path in GROUP_SAFE_CALIBRATION_REQUIRED
    ])
group_safe_ready = all(path.exists() and path.stat().st_size > 0 for path in GROUP_SAFE_CALIBRATION_REQUIRED)
group_safe_should_run = RUN_GROUP_SAFE_SCORE_CALIBRATION is True or (
    str(RUN_GROUP_SAFE_SCORE_CALIBRATION).lower() == 'auto'
)
group_safe_prerequisites = [
    Path('reports/revision/experimental/external/ptbxl/ptbxl_full_predictions.npz'),
    Path('reports/revision/experimental/external/ptbxl/ptbxl_full_fold9_predictions.npz'),
    Path('reports/revision/metrics/external_ptbxl_protocol_gate.json'),
]
group_safe_prerequisites_ready = all(path.exists() and path.stat().st_size > 0 for path in group_safe_prerequisites)
if not group_safe_prerequisites_ready:
    if RUN_GROUP_SAFE_SCORE_CALIBRATION is True:
        raise FileNotFoundError('Group-safe PTB-XL calibration was forced, but prerequisites are missing: ' + '; '.join(str(path) for path in group_safe_prerequisites if not path.exists() or path.stat().st_size == 0))
    group_safe_should_run = False
    print('Deferred group-safe PTB-XL calibration until fold-9 and fold-10 Full predictions plus the PTB-XL gate exist.')
if group_safe_should_run:
    run(
        'python -u scripts/revision/33_group_safe_score_calibration.py '
        f'--dataset ptbxl --threshold 0.5 --n-bins 15 --primary-fraction {GROUP_SAFE_CALIBRATION_PRIMARY_FRACTION} '
        f'--n-boot {GROUP_SAFE_CALIBRATION_N_BOOT} --strict --reuse-existing --analysis-lock "{PTBXL_ADAPTATION_LOCK}" '
        f'--metric-cache-dir "{GROUP_SAFE_CALIBRATION_CACHE_DIR}"',
        log_path='reports/revision/logs/group_safe_score_calibration_ptbxl.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{DRIVE_ROOT / "revision_artifacts" / "reports" / "revision"}"',
        log_path='reports/revision/logs/group_safe_score_calibration_ptbxl_mirror_publish.log',
    )
else:
    print('Reusing group-safe PTB-XL score-calibration package.' if group_safe_ready else 'Group-safe PTB-XL calibration deferred; outputs are missing or stale.')
for path in GROUP_SAFE_CALIBRATION_REQUIRED:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## External Frozen-Encoder Representation Extraction

Protocol v2 binds every fold cache to the source prediction manifest, current dataset archive, loader contract, canonical OOF contract, and exact checkpoint SHA256. The first v2 pass hashes the dataset ZIP once and stores a fingerprint-keyed SHA256 sidecar beside the fold caches. Record embeddings are arithmetic means of pre-classifier slice embeddings; they are not Q=3 probability aggregates. Existing v1 representation caches are intentionally regenerated once.


In [ ]:
# GPU + Mamba runtime required. Source-bound v2 fold caches live on canonical Drive and survive disconnects.
RUN_EXTERNAL_REPRESENTATION_EXTRACTION = 'auto'
EXTERNAL_REPRESENTATION_MODELS = 'full,resnet,raw_mamba'
EXTERNAL_REPRESENTATION_BATCH_SIZE = 128
EXTERNAL_REPRESENTATION_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'predictions' / 'external_representation_folds'
EXTERNAL_REPRESENTATION_CHECKPOINT_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'experimental'

def _representation_checkpoint_dir(name):
    candidates = [
        EXTERNAL_REPRESENTATION_CHECKPOINT_ROOT / name,
        Path('reports/revision/experimental') / name,
    ]
    return next((path for path in candidates if path.exists()), candidates[0])

external_resnet_checkpoint_dir = _representation_checkpoint_dir('resnet1d_cnn_checkpoints')
external_raw_mamba_checkpoint_dir = _representation_checkpoint_dir('raw_mamba_checkpoints')
external_transformer_checkpoint_dir = _representation_checkpoint_dir('transformer_ecg_checkpoints')
if '_restore_report_artifact' in globals():
    representation_contract_relpaths = [
        Path('metrics/resnet1d_cnn_baseline_summary.json'),
        Path('manifests/resnet1d_cnn_baseline_manifest.json'),
        Path('metrics/paired_full_vs_resnet_comparison.json'),
        Path('metrics/raw_mamba_baseline_summary.json'),
        Path('manifests/raw_mamba_baseline_manifest.json'),
        Path('metrics/paired_full_vs_raw_mamba_comparison.json'),
    ]
    representation_restore_roots = globals().get('restore_source_roots', [
        DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision',
    ])
    print('Representation in-domain contract restore:', [
        _restore_report_artifact(Path('reports/revision') / rel, representation_restore_roots)
        for rel in representation_contract_relpaths
    ])
external_rep_models = [item.strip() for item in EXTERNAL_REPRESENTATION_MODELS.split(',') if item.strip()]
transformer_rep_prerequisites = [
    Path('reports/revision/metrics/transformer_ecg_baseline_summary.json'),
    Path('reports/revision/manifests/transformer_ecg_baseline_manifest.json'),
    Path('reports/revision/metrics/paired_full_vs_transformer_comparison.json'),
] + [
    external_transformer_checkpoint_dir / f'fold{fold}_transformer_ecg_final.pt' for fold in range(1, 6)
]
if (
    all(path.exists() and path.stat().st_size > 0 for path in transformer_rep_prerequisites)
    and bool(globals().get('transformer_checkpoint_contract_ready', False))
):
    external_rep_models.append('transformer')
    print('Transformer representation extraction enabled after its in-domain gate.')
def _embedding_path(model, tag=''):
    stem = {'full': 'ecg_ramba_full', 'resnet': 'resnet1d_cnn', 'raw_mamba': 'raw_mamba', 'transformer': 'transformer_ecg'}[model]
    suffix = f'_{tag}' if tag else ''
    return Path(f'reports/revision/predictions/external_ptbxl_{stem}{suffix}_record_embeddings.npz')
def _embedding_manifest_path(model, tag=''):
    stem = {'full': 'ecg_ramba_full', 'resnet': 'resnet1d_cnn', 'raw_mamba': 'raw_mamba', 'transformer': 'transformer_ecg'}[model]
    suffix = f'_{tag}' if tag else ''
    return Path(f'reports/revision/manifests/external_ptbxl_{stem}{suffix}_embedding_manifest.json')
representation_required = (
    [_embedding_path(model) for model in external_rep_models]
    + [_embedding_path(model, 'fold9') for model in external_rep_models]
    + [_embedding_manifest_path(model) for model in external_rep_models]
    + [_embedding_manifest_path(model, 'fold9') for model in external_rep_models]
)
if '_restore_report_artifact' in globals():
    representation_restore_roots = globals().get('restore_source_roots', [MIRROR_REVISION_ROOT])
    representation_restore = [
        _restore_report_artifact(path, representation_restore_roots)
        for path in representation_required
    ]
    restored = sum(row.get('status') == 'restored' for row in representation_restore)
    print(f'External representation targeted restore: restored={restored} total={len(representation_restore)}')
representation_ready = all(path.exists() and path.stat().st_size > 0 for path in representation_required)
representation_should_run = RUN_EXTERNAL_REPRESENTATION_EXTRACTION is True or (
    str(RUN_EXTERNAL_REPRESENTATION_EXTRACTION).lower() == 'auto'
)
representation_source_predictions = [
    Path(f'reports/revision/experimental/external/ptbxl/ptbxl_{stem}{suffix}_predictions.npz')
    for stem in [{'full': 'full', 'resnet': 'resnet1d_cnn', 'raw_mamba': 'raw_mamba', 'transformer': 'transformer_ecg'}[model] for model in external_rep_models]
    for suffix in ['', '_fold9']
]
representation_prerequisites_ready = bool(globals().get('learned_in_domain_ready', False)) and all(
    path.exists() and path.stat().st_size > 0 for path in representation_source_predictions
)
if not representation_prerequisites_ready:
    if RUN_EXTERNAL_REPRESENTATION_EXTRACTION is True:
        missing = [str(path) for path in representation_source_predictions if not path.exists() or path.stat().st_size == 0]
        if not bool(globals().get('learned_in_domain_ready', False)):
            missing.append('Notebook 04 ResNet/Raw-Mamba paired OOF/checkpoint prerequisites')
        raise FileNotFoundError('External representation extraction was forced, but prerequisite artifacts are missing: ' + '; '.join(missing))
    representation_should_run = False
    print('Deferred external frozen-encoder representation extraction until Notebook 04 and the PTB-XL comparator test/fold-9 predictions are complete.')
if representation_should_run and not representation_ready:
    try:
        require_gpu_inference_runtime('External frozen-encoder representation extraction')
    except RuntimeError as exc:
        if RUN_EXTERNAL_REPRESENTATION_EXTRACTION is True:
            raise
        representation_should_run = False
        print('DEFERRED external representation extraction:', exc)
        print('Reconnect an A100 High-RAM runtime; source-bound per-fold Drive caches will be reused.')
if representation_should_run:
    extract_base = (
        'python -u scripts/revision/34_extract_external_representations.py '
        f'--dataset ptbxl --models {",".join(external_rep_models)} --checkpoint-kind final_ema '
        f'--batch-size {EXTERNAL_REPRESENTATION_BATCH_SIZE} --num-workers 0 --device cuda --amp --allow-tf32 '
        f'--reuse-existing --fold-cache-dir "{EXTERNAL_REPRESENTATION_CACHE_DIR}" '
        f'--resnet-checkpoint-dir "{external_resnet_checkpoint_dir}" '
        f'--raw-mamba-checkpoint-dir "{external_raw_mamba_checkpoint_dir}" '
        f'--transformer-checkpoint-dir "{external_transformer_checkpoint_dir}"'
    )
    run(extract_base, log_path='reports/revision/logs/external_ptbxl_representation_test.log')
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{DRIVE_ROOT / "revision_artifacts" / "reports" / "revision"}"',
        log_path='reports/revision/logs/external_ptbxl_representation_test_mirror_publish.log',
    )
    run(extract_base + ' --ptbxl-folds 9 --output-tag fold9', log_path='reports/revision/logs/external_ptbxl_representation_fold9.log')
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{DRIVE_ROOT / "revision_artifacts" / "reports" / "revision"}"',
        log_path='reports/revision/logs/external_ptbxl_representation_fold9_mirror_publish.log',
    )
else:
    print('Reusing external PTB-XL frozen-encoder representations.' if representation_ready else 'External PTB-XL representation extraction deferred; outputs are missing or stale.')
for path in representation_required:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## True Few-Shot Frozen-Encoder Head Adaptation

Fractions 1/5/10% denote nested fractions of independent labeled target groups, not a fixed number of examples per class. Five fold-specific logistic heads are fitted on train-only standardized, mean-pooled record embeddings and then probability-averaged. Encoder weights remain frozen.


In [ ]:
# CPU-only once representation NPZs exist. This fits new linear classifier parameters; encoders stay frozen.
RUN_TRUE_FEWSHOT_HEAD_ADAPTATION = 'auto'
TRUE_FEWSHOT_N_BOOT = 1000
TRUE_FEWSHOT_PRIMARY_FRACTION = 0.10
TRUE_FEWSHOT_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'predictions' / 'fewshot_head_adaptation_cache'
TRUE_FEWSHOT_METRIC_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'metrics' / 'true_fewshot_head_metric_cache'
TRUE_FEWSHOT_REQUIRED = [
    Path('reports/revision/metrics/true_fewshot_head_ptbxl_summary.csv'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl.csv'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl_paired.csv'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl_primary.csv'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl_learning_curve.csv'),
    Path('reports/revision/figures/figure_true_fewshot_head_ptbxl_learning_curve.png'),
    Path('reports/revision/metrics/true_fewshot_head_ptbxl_bootstrap.json'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl_coefficients.csv'),
    Path('reports/revision/manifests/true_fewshot_head_ptbxl_splits.npz'),
    Path('reports/revision/manifests/true_fewshot_head_ptbxl_manifest.json'),
]
if '_restore_report_artifact' in globals():
    true_fewshot_restore_roots = globals().get('restore_source_roots', [
        DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision',
    ])
    print('True few-shot result targeted restore:', [
        _restore_report_artifact(path, true_fewshot_restore_roots, remove_unpublished_active=True)
        for path in TRUE_FEWSHOT_REQUIRED
    ])
true_fewshot_ready = all(path.exists() and path.stat().st_size > 0 for path in TRUE_FEWSHOT_REQUIRED)
true_fewshot_should_run = RUN_TRUE_FEWSHOT_HEAD_ADAPTATION is True or (
    str(RUN_TRUE_FEWSHOT_HEAD_ADAPTATION).lower() == 'auto'
)
true_fewshot_models = list(globals().get('external_rep_models', []))
true_fewshot_embedding_paths = list(globals().get('representation_required', []))
true_fewshot_prerequisites = true_fewshot_embedding_paths + [
    Path('reports/revision/manifests/external_comparator_paired_manifest.json'),
    Path('reports/revision/metrics/external_ptbxl_protocol_gate.json'),
]
true_fewshot_prerequisites_ready = bool(true_fewshot_models) and all(
    path.exists() and path.stat().st_size > 0 for path in true_fewshot_prerequisites
)
if not true_fewshot_prerequisites_ready:
    if RUN_TRUE_FEWSHOT_HEAD_ADAPTATION is True:
        missing = [str(path) for path in true_fewshot_prerequisites if not path.exists() or path.stat().st_size == 0]
        raise FileNotFoundError('True few-shot head adaptation was forced, but prerequisite artifacts are missing: ' + '; '.join(missing))
    true_fewshot_should_run = False
    print('Deferred true few-shot head adaptation until external representations and paired external audit are complete.')
if true_fewshot_should_run:
    run(
        'python -u scripts/revision/35_true_fewshot_head_adaptation.py '
        f'--dataset ptbxl --models {",".join(true_fewshot_models)} --fractions 0,0.01,0.05,0.10 --primary-fraction {TRUE_FEWSHOT_PRIMARY_FRACTION} '
        '--seeds 42,43,44,45,46 --threshold 0.5 --n-bins 15 '
        f'--n-boot {TRUE_FEWSHOT_N_BOOT} --strict --reuse-existing --analysis-lock "{PTBXL_ADAPTATION_LOCK}" '
        f'--cache-dir "{TRUE_FEWSHOT_CACHE_DIR}" --metric-cache-dir "{TRUE_FEWSHOT_METRIC_CACHE_DIR}"',
        log_path='reports/revision/logs/true_fewshot_head_ptbxl.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-prefix predictions/fewshot_head_adaptation_cache --refresh-existing-prefix metrics/true_fewshot_head_metric_cache --mirror-root "{DRIVE_ROOT / "revision_artifacts" / "reports" / "revision"}"',
        log_path='reports/revision/logs/true_fewshot_head_ptbxl_mirror_publish.log',
    )
else:
    print('Reusing true few-shot PTB-XL frozen-encoder-head package.' if true_fewshot_ready else 'True few-shot PTB-XL head adaptation deferred; outputs are missing or stale.')
for path in TRUE_FEWSHOT_REQUIRED:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## Reviewer Evidence Completion Matrix And Inventory

In [ ]:
# Reviewer-evidence completion matrix. Incomplete stages remain explicit instead of silently claim-ready.
from scripts.revision.common import save_csv, save_json

def _nonempty(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0

def _stage_row(stage, reviewer_items, paths, hardware, next_action, semantic_ready=None):
    paths = [Path(path) for path in paths]
    present = sum(_nonempty(path) for path in paths)
    files_ready = bool(paths) and present == len(paths)
    ready = files_ready if semantic_ready is None else bool(semantic_ready) and files_ready
    return {
        'stage': stage, 'reviewer_items': reviewer_items, 'ready': ready,
        'present_artifacts': present, 'required_artifacts': len(paths), 'hardware': hardware,
        'next_action_if_incomplete': '' if ready else next_action,
    }

def _gate_passed(dataset):
    path = Path(f'reports/revision/metrics/external_{dataset}_protocol_gate.json')
    if not _nonempty(path):
        return False
    payload = json.loads(path.read_text(encoding='utf-8'))
    return payload.get('status') == 'protocol_gate_passed' and payload.get('protocol_gate_passed') is True

reviewer_external_datasets = ['ptbxl', 'georgia', 'cpsc2021']
reviewer_external_models = ['resnet', 'raw_mamba', 'transformer']
gate_paths = [Path(f'reports/revision/metrics/external_{dataset}_protocol_gate.json') for dataset in reviewer_external_datasets]
transformer_contract_paths = globals().get('transformer_prerequisites', [])
transformer_contract_ready = bool(transformer_contract_paths) and all(_nonempty(path) for path in transformer_contract_paths) and bool(globals().get('transformer_checkpoint_contract_ready', False))
external_test_expected = [path for dataset in reviewer_external_datasets for model in reviewer_external_models for path in _comparator_artifacts(dataset, model)]
external_fold9_expected = [path for model in reviewer_external_models for path in _comparator_artifacts('ptbxl', model, 'fold9')]
representation_expected = (
    [_embedding_path(model) for model in ['full', *reviewer_external_models]]
    + [_embedding_path(model, 'fold9') for model in ['full', *reviewer_external_models]]
    + [_embedding_manifest_path(model) for model in ['full', *reviewer_external_models]]
    + [_embedding_manifest_path(model, 'fold9') for model in ['full', *reviewer_external_models]]
)
paired_coverage_ready = False
paired_summary_path = Path('reports/revision/metrics/external_comparator_paired_summary.json')
if _nonempty(paired_summary_path):
    paired_payload = json.loads(paired_summary_path.read_text(encoding='utf-8'))
    paired_pairs = {(str(row.get('dataset')), str(row.get('comparator'))) for row in paired_payload.get('rows', [])}
    paired_coverage_ready = paired_payload.get('status') == 'complete' and {(dataset, model) for dataset in reviewer_external_datasets for model in reviewer_external_models}.issubset(paired_pairs)
true_fewshot_coverage_ready = False
true_fewshot_primary_path = Path('reports/revision/tables/table_true_fewshot_head_ptbxl_primary.csv')
if _nonempty(true_fewshot_primary_path):
    primary_df = pd.read_csv(true_fewshot_primary_path)
    true_fewshot_coverage_ready = {'full', *reviewer_external_models}.issubset(set(primary_df.get('model', [])))

reviewer_stage_rows = [
    _stage_row('canonical_chapman_oof', 'shared contract', [Path('reports/revision/predictions/oof_final_ema_predictions.npz'), Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json')], 'A100 only if stale', 'Run Generate OOF Predictions on A100.'),
    _stage_row('dataset_specific_external_gates', 'R1-C4/R1-C5', gate_paths, 'CPU', 'Run External Protocol Gate.', semantic_ready=all(_gate_passed(dataset) for dataset in reviewer_external_datasets)),
    _stage_row('ptbxl_fold9_adaptation_pool', 'R2-C4', globals().get('PTBXL_FOLD9_REQUIRED', []), 'A100', 'Run PTB-XL Fold 9 Adaptation-Pool Export.'),
    _stage_row('notebook04_learned_comparator_contracts', 'R1-C4', globals().get('learned_in_domain_prerequisites', []) + globals().get('transformer_prerequisites', []), 'A100 in Notebook 04', 'Complete ResNet, Raw Mamba, and Transformer OOF/paired/checkpoint stages in Notebook 04.', semantic_ready=bool(globals().get('learned_in_domain_ready', False)) and transformer_contract_ready),
    _stage_row('external_zero_target_label_comparators', 'R1-C4/R1-C5', external_test_expected, 'A100', 'Run External Learned-Comparator Zero-Target-Label Inference.'),
    _stage_row('ptbxl_fold9_comparator_predictions', 'R2-C4', external_fold9_expected, 'A100', 'Run PTB-XL fold 9 learned-comparator inference.'),
    _stage_row('ptbxl_fold_protocol_audit', 'R1-C5/R2-C4', globals().get('PTBXL_FOLD_PROTOCOL_REQUIRED', []), 'CPU', 'Run PTB-XL Fold 9/10 Patient and Unsupported-Only Audit.'),
    _stage_row('paired_external_group_bootstrap', 'R1-C5', globals().get('EXTERNAL_PAIRED_REQUIRED', []), 'CPU', 'Run Paired External Comparator Audit.', semantic_ready=paired_coverage_ready),
    _stage_row('group_safe_score_calibration', 'R2-C4 sensitivity', globals().get('GROUP_SAFE_CALIBRATION_REQUIRED', []), 'CPU', 'Run Group-Safe Score Calibration.'),
    _stage_row('external_frozen_encoder_representations', 'R2-C4', representation_expected, 'A100', 'Run External Frozen-Encoder Representation Extraction.'),
    _stage_row('true_fewshot_frozen_encoder_heads', 'R2-C4 primary', globals().get('TRUE_FEWSHOT_REQUIRED', []), 'CPU', 'Run True Few-Shot Frozen-Encoder Head Adaptation.', semantic_ready=true_fewshot_coverage_ready),
]
reviewer_status_df = pd.DataFrame(reviewer_stage_rows)
display(reviewer_status_df)
reviewer_status_table = Path('reports/revision/tables/table_notebook02_reviewer_evidence_status.csv')
reviewer_status_json = Path('reports/revision/metrics/notebook02_reviewer_evidence_status.json')
save_csv(reviewer_status_table, reviewer_stage_rows)
save_json(reviewer_status_json, {
    'status': 'complete' if bool(reviewer_status_df['ready'].all()) else 'incomplete',
    'all_reviewer_external_stages_ready': bool(reviewer_status_df['ready'].all()),
    'canonical_artifact_root': str(DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'),
    'stages': reviewer_stage_rows,
    'claim_boundary': 'Report every dataset, model, metric, and adaptation protocol separately; no broad superiority claim.',
})
print('Wrote:', reviewer_status_table)
print('Wrote:', reviewer_status_json)

run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/predictions_external_artifact_inventory.log',
)


## Mirror Artifacts To Stable Drive Cache

In [ ]:
run(
    'python -u scripts/revision/06_freeze_oof.py '
    '--record-file reports/revision/predictions/oof_final_ema_predictions.npz '
    '--slice-file reports/revision/predictions/oof_final_ema_slice_predictions.npz '
    '--summary-file reports/revision/metrics/oof_final_ema_prediction_summary.json '
    '--class-table reports/revision/tables/oof_final_ema_class_summary.csv '
    '--run-manifest reports/revision/manifests/oof_final_ema_prediction_run_manifest.json '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-records 44186 --expected-folds 5 --q 3 --expected-checkpoint-kind final_ema --manuscript-ready-strict --group-sidecar reports/revision/manifests/oof_final_ema_group_sidecar.npz --check-existing-freeze',
    log_path='reports/revision/logs/oof_final_ema_pre_publish_contract.log',
)

source_root = Path('reports/revision')
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/oof_mirror_publish.log',
)
